# FaultLines run: Gemma 4 on the competition's dev tasks

Runs a local Gemma 4 agent on N of the 129 dev tasks with **faultlines-mini**. It's a re-implementation of the (unreleased) `swegemma` harness, built from the competition's `HARNESS_README.md`: same 9 tools, prompt sections, nudges, limits and pytest verification. It also saves the model's **raw text** for every turn. Then it times the runs and labels them.

**Before you run (right-hand panel):**
1. **Settings → Accelerator:** GPU T4 x2 (or better). **Internet:** On.
2. **Add Input → Competitions:** `gemma-4-developer-agent`.
3. **Add Input → Models:** Gemma 4 → Transformers → `gemma-4-e4b-it`. It's the variant that fits a T4. The leaderboard model (31B W4A16) needs Ampere-class GPUs.

**Stages:**
1. **Gold replay (no GPU).** Applies each task's reference fix to check the environment and measure setup and verify time. It should resolve about 100%.
2. **Start vLLM and check the chat template.** Look at the printed output once.
3. **Run the model on N tasks,** then show timing, the projected runs per week, and the labels.

Outputs go to `/kaggle/working/fl_runs/`: `runs.jsonl`, `labeled.jsonl`, patches and the vLLM log.

## 0. Write the FaultLines package

In [ ]:
!mkdir -p faultlines/src/faultlines faultlines/src/faultlines/atlas faultlines/src/faultlines/harness faultlines/src/faultlines/toolguard

In [ ]:
%%writefile faultlines/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "faultlines"
version = "0.1.0"
description = "Where local Gemma 4 coding agents break: tool-call guard + failure atlas"
readme = "README.md"
requires-python = ">=3.10"
license = {text = "Apache-2.0"}
dependencies = []

[project.optional-dependencies]
dev = ["pytest>=7"]

[project.scripts]
faultlines = "faultlines.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.setuptools.package-data]
"faultlines.toolguard" = ["harness_tools.json"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]


In [ ]:
%%writefile faultlines/src/faultlines/__init__.py
"""FaultLines: where local Gemma 4 coding agents break, and how much of it is the interface."""

__version__ = "0.1.0"


In [ ]:
%%writefile faultlines/src/faultlines/atlas/__init__.py
"""Failure atlas: trace schema, taxonomy, auto-labeler, reports."""

from .autolabel import Thresholds, label_run, patch_files
from .taxonomy import DEFINITIONS, GROUPS, Primary, group_of
from .trace import Run, Step, read_runs, write_runs

__all__ = ["Thresholds", "label_run", "patch_files", "DEFINITIONS", "GROUPS", "Primary", "group_of",
           "Run", "Step", "read_runs", "write_runs"]


In [ ]:
%%writefile faultlines/src/faultlines/atlas/adapters.py
"""Convert harness output into FaultLines `Run` records.

`swegemma eval` writes (per the third-party copy of the harness spec):
    results/summary.json          aggregate resolution rate
    results/task_results.jsonl    per-task metrics
    results/patches/              extracted diffs
    results/test_outputs/         pytest logs
    results/traces/               ATIF-compatible SessionTrace JSON

!! Field names below are BEST GUESSES. In week 1, open one real trace and one
!! task_results line, then fix the `_pick(...)` key lists. Everything else in
!! the pipeline only depends on the `Run` schema, so this is the only file to edit.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Iterator

from .autolabel import patch_files
from .trace import Run, Step


def _pick(d: dict[str, Any], *keys: str, default: Any = None) -> Any:
    for k in keys:
        if isinstance(d, dict) and k in d and d[k] is not None:
            return d[k]
    return default


def _as_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, list):  # list of content parts
        return "\n".join(_as_text(p.get("text", p) if isinstance(p, dict) else p) for p in x)
    if isinstance(x, dict):
        return _as_text(_pick(x, "text", "content", "output", default=json.dumps(x)))
    return str(x)


def _args(x: Any) -> dict[str, Any]:
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            v = json.loads(x)
            return v if isinstance(v, dict) else {"_raw": x}
        except json.JSONDecodeError:
            return {"_raw": x}
    return {}


def atif_to_steps(trace: dict[str, Any]) -> list[Step]:
    """Map an ATIF-style trace to agent turns. Tolerates several key spellings."""
    raw_steps = _pick(trace, "steps", "events", "turns", default=[])
    steps: list[Step] = []
    pending: Step | None = None
    for st in raw_steps:
        source = str(_pick(st, "source", "role", "author", default="")).lower()
        if source in ("agent", "assistant", "model"):
            text = _as_text(_pick(st, "raw_output", "message", "content", "text"))
            reasoning = _as_text(_pick(st, "reasoning_content", "thinking"))
            calls = _pick(st, "tool_calls", "function_calls", default=[]) or []
            first = calls[0] if calls else None
            step = Step(
                index=len(steps),
                raw_output=(reasoning + "\n" + text).strip() if reasoning else text,
                tool_name=_pick(first, "function_name", "name", "tool_name") if first else None,
                tool_args=_args(_pick(first, "arguments", "args", "parameters")) if first else None,
                finish_reason=_pick(st, "finish_reason", "stop_reason"),
            )
            metrics = _pick(st, "metrics", "usage", default={}) or {}
            step.prompt_tokens = _pick(metrics, "prompt_tokens", "input_tokens")
            step.completion_tokens = _pick(metrics, "completion_tokens", "output_tokens")
            obs = _pick(st, "observation", "tool_results")
            if obs is not None:
                results = _pick(obs, "results", default=obs) if isinstance(obs, dict) else obs
                step.tool_result = _as_text(results[0] if isinstance(results, list) and results else results)
            steps.append(step)
            pending = step
        elif source in ("tool", "function", "observation") and pending is not None:
            pending.tool_result = _as_text(_pick(st, "content", "message", "output", "text"))
        elif source in ("system", "user", "harness"):
            text = _as_text(_pick(st, "message", "content", "text")).lower()
            if pending is not None:
                # GUESS: tune these phrases to the harness's real compaction/nudge messages.
                if "compact" in text or ("summar" in text and "history" in text):
                    pending.events.append("compaction")
                if ("continue" in text or "tool call" in text) and "nudge" not in pending.events:
                    pending.events.append("nudge")
    return steps


def load_swegemma_results(results_dir: str | Path, tasks_path: str | Path | None = None,
                          model: str = "", config_hash: str = "", context_limit: int | None = 32768) -> Iterator[Run]:
    """Join task_results.jsonl + traces/ + patches/ + test_outputs/ (+ tasks.jsonl for gold files)."""
    rd = Path(results_dir)
    gold: dict[str, list[str]] = {}
    if tasks_path:
        for line in Path(tasks_path).read_text().splitlines():
            if line.strip():
                t = json.loads(line)
                tid = str(_pick(t, "instance_id", "task_id", "id"))
                gold[tid] = patch_files(_pick(t, "patch", "gold_patch", default=""))

    for line in (rd / "task_results.jsonl").read_text().splitlines():
        if not line.strip():
            continue
        tr = json.loads(line)
        tid = str(_pick(tr, "instance_id", "task_id", "id"))
        run = Run(
            task_id=tid,
            model=model or str(_pick(tr, "model", default="")),
            config_hash=config_hash,
            resolved=_pick(tr, "resolved", "success"),
            exit_reason=str(_pick(tr, "exit_reason", "termination_reason", "stop_reason", default="")),
            patch_applied=_pick(tr, "patch_applied", "applied"),
            gold_files=gold.get(tid, []),
            context_limit=context_limit,
        )
        for cand in (rd / "patches" / f"{tid}.diff", rd / "patches" / f"{tid}.patch"):
            if cand.exists():
                run.agent_patch = cand.read_text()
        for cand in (rd / "test_outputs" / f"{tid}.log", rd / "test_outputs" / f"{tid}.txt"):
            if cand.exists():
                run.test_output = cand.read_text()[-20000:]
        tfile = next(iter(sorted((rd / "traces").glob(f"*{tid}*.json"))), None) if (rd / "traces").exists() else None
        if tfile:
            run.steps = atif_to_steps(json.loads(tfile.read_text()))
        if not run.exit_reason:
            run.exit_reason = "submitted" if any(s.tool_name == "submit_patch" for s in run.steps) else "unknown"
        yield run


In [ ]:
%%writefile faultlines/src/faultlines/atlas/autolabel.py
"""Heuristic labeler: fills step labels and one primary label per run.

Thresholds are deliberately simple and named so the paper can state them and
the kappa check (vs. hand labels) can tune them. Tune on real traces in week 3.
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass

from ..toolguard import ToolGuard, ToolRegistry, find_loops
from ..toolguard.gemma_syntax import CALL_OPEN
from ..toolguard.repair import ToolCall
from .taxonomy import Primary
from .trace import Run

GRAPH_TOOLS = {"get_code_neighbors", "search_similar_code", "get_code_subgraph"}

RUNTIME_PATTERNS = [
    ("EDIT_NO_MATCH", re.compile(r"(no match|not found in file|did not match|could not find .*(string|text)|"
                                 r"old_str .*not found|0 occurrences)", re.I)),
    ("FILE_NOT_FOUND", re.compile(r"(no such file|filenotfounderror|does not exist|file not found)", re.I)),
    ("TIMEOUT", re.compile(r"(timed out|timeout expired|command timeout)", re.I)),
    ("PERMISSION", re.compile(r"(permission denied|path traversal|outside (the )?workspace)", re.I)),
    ("TOOL_ERROR", re.compile(r"^\s*(error|exception|traceback)\b", re.I)),
]

BROKEN_TEST_PATTERNS = re.compile(
    r"(error collecting|errors? during collection|SyntaxError|IndentationError|ImportError while importing|"
    r"ModuleNotFoundError|patch does not apply|failed to apply)", re.I)


@dataclass
class Thresholds:
    tail_window: int = 5          # INTERFACE: look at the last N turns
    tail_bad: int = 3             # INTERFACE: ... at least this many bad turns
    high_invalid_rate: float = 0.2
    context_ratio: float = 0.9
    loop_tail_fraction: float = 1 / 3
    runtime_errors_flag: int = 3


def patch_files(patch: str) -> list[str]:
    files = []
    for m in re.finditer(r"^diff --git a/(\S+) b/(\S+)", patch or "", re.M):
        files.append(m.group(2))
    if not files:
        files = [m.group(1) for m in re.finditer(r"^\+\+\+ b/(\S+)", patch or "", re.M)]
    return sorted(set(files))


# swegemma tools reply with JSON: {"status": "error", "error_type": ..., "error_message": ...}
ERROR_TYPE_MAP = {
    "FileEditError": "EDIT_FAILED",          # old_string not found / not unique / empty file
    "TimeoutExceeded": "TIMEOUT",
    "ValidationError": "BAD_PATH",           # e.g. '..' traversal
    "CommandError": "COMMAND_FAILED",        # non-zero exit; often expected (failing repro test)
    "BudgetExceeded": "BUDGET",
}
NOT_A_RUNTIME_ERROR = {"COMMAND_FAILED"}   # a failing test/grep is normal agent behaviour


def classify_runtime(tool_result: str) -> str | None:
    text = (tool_result or "").strip()
    if text.startswith("{"):
        try:
            obj = json.loads(text)
        except json.JSONDecodeError:
            obj = None
        if isinstance(obj, dict):
            if obj.get("status") != "error":
                return None
            et = str(obj.get("error_type", "TOOL_ERROR"))
            msg = str(obj.get("error_message", "")).lower()
            if et == "FileEditError" and ("not found" in msg or "no match" in msg):
                return "EDIT_NO_MATCH"
            if "no such file" in msg or "does not exist" in msg or "not exist" in msg:
                return "FILE_NOT_FOUND"
            return ERROR_TYPE_MAP.get(et, f"TOOL_{et}")
    head = text[:600]
    for code, pat in RUNTIME_PATTERNS:
        if pat.search(head):
            return code
    return None


def label_steps(run: Run, registry: ToolRegistry) -> None:
    guard = ToolGuard(registry, detect_loops=False)
    for s in run.steps:
        text = s.raw_output or ""
        if CALL_OPEN in text:
            res = guard.check(text)
        elif s.tool_name:
            # Trace kept only the parsed call: judge it at schema level.
            res = guard.check_structured(s.tool_name, s.tool_args or {})
        else:
            res = guard.check("")  # no call at all
        s.call_status = res.status
        s.call_issues = sorted({i.code.value for i in res.issues})
        s.runtime_error = classify_runtime(s.tool_result) if s.tool_name else None


def _executed_key(s) -> str | None:
    if not s.tool_name:
        return None
    return ToolCall(s.tool_name, s.tool_args or {}).key()


def label_run(run: Run, registry: ToolRegistry, th: Thresholds | None = None) -> Run:
    th = th or Thresholds()
    label_steps(run, registry)
    steps = run.steps
    n = len(steps)
    ev: list[str] = []
    sec: list[str] = []

    bad = [s.call_status in ("rejected", "no_call") for s in steps]
    tail = bad[-th.tail_window:]
    last_truncated = bool(steps) and "TRUNCATED_CALL" in steps[-1].call_issues and steps[-1].call_status == "rejected"
    interface_breakdown = sum(tail) >= th.tail_bad or last_truncated
    invalid_rate = (sum(bad) / n) if n else 0.0
    if invalid_rate >= th.high_invalid_rate:
        sec.append("HIGH_INVALID_RATE")
    if any("TRUNCATED_CALL" in s.call_issues for s in steps):
        sec.append("HAD_TRUNCATION")

    loops = find_loops([_executed_key(s) for s in steps])
    if loops:
        sec.append("HAD_LOOP")
    loop_late = any(i >= n * (1 - th.loop_tail_fraction) for i, _ in loops)

    max_prompt = max((s.prompt_tokens or 0 for s in steps), default=0)
    compactions = sum("compaction" in e for s in steps for e in s.events)
    context_pressure = bool(run.context_limit and max_prompt >= th.context_ratio * run.context_limit) or compactions > 0
    if context_pressure:
        sec.append("CONTEXT_PRESSURE")

    rt = sum(1 for s in steps if s.runtime_error and s.runtime_error not in NOT_A_RUNTIME_ERROR)
    if rt >= th.runtime_errors_flag:
        sec.append("RUNTIME_ERRORS")
    if any(s.tool_name in GRAPH_TOOLS for s in steps):
        sec.append("USED_CODE_GRAPH")
    if any("nudge" in e for s in steps for e in s.events):
        sec.append("NUDGED")

    ev.append(f"turns={n} invalid_rate={invalid_rate:.2f} tail_bad={sum(tail)}/{len(tail)} "
              f"loops={len(loops)} max_prompt={max_prompt} compactions={compactions} runtime_errors={rt}")

    submitted = run.exit_reason == "submitted"
    if run.exit_reason == "error":
        primary = Primary.ENV_ERROR
        ev.append("exit_reason=error")
    elif run.resolved is True:
        primary = Primary.RESOLVED
    elif run.resolved is None:
        primary = Primary.UNKNOWN_OUTCOME
    elif not submitted:
        if interface_breakdown:
            primary = Primary.INTERFACE
            ev.append("last turn truncated" if last_truncated else f"{sum(tail)} bad calls in last {len(tail)} turns")
        elif loop_late:
            primary = Primary.LOOP
            i, sig = loops[-1]
            ev.append(f"{sig.kind} loop (period {sig.period}) at turn {i}")
        elif context_pressure:
            primary = Primary.CONTEXT
            ev.append(f"max_prompt={max_prompt} limit={run.context_limit} compactions={compactions}")
        else:
            primary = Primary.BUDGET
            ev.append(f"exit_reason={run.exit_reason or 'unknown'}")
    else:
        pf = patch_files(run.agent_patch)
        if not run.agent_patch.strip() or not pf:
            primary = Primary.NO_PATCH
        elif run.patch_applied is False or BROKEN_TEST_PATTERNS.search(run.test_output or ""):
            primary = Primary.BROKEN_PATCH
            m = BROKEN_TEST_PATTERNS.search(run.test_output or "")
            ev.append(f"test output: {m.group(0)}" if m else "patch_applied=False")
        elif run.gold_files:
            overlap = set(pf) & set(run.gold_files)
            if overlap:
                primary = Primary.WRONG_FIX
                ev.append(f"touched gold file(s) {sorted(overlap)}")
            else:
                primary = Primary.WRONG_LOCATION
                ev.append(f"patched {pf}, gold {run.gold_files}")
        else:
            primary = Primary.UNRESOLVED_OTHER

    run.primary_failure = primary.value
    run.secondary = sec
    run.evidence = ev
    return run


In [ ]:
%%writefile faultlines/src/faultlines/atlas/report.py
"""Statistics and markdown tables for the paper (RQ1, RQ2, RQ4, label agreement)."""

from __future__ import annotations

import math
from collections import Counter, defaultdict
from typing import Iterable

from .taxonomy import group_of
from .trace import Run


def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """Wilson score 95% interval for a proportion k/n."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


def cohen_kappa(pairs: Iterable[tuple[str, str]]) -> float:
    pairs = list(pairs)
    n = len(pairs)
    if n == 0:
        return float("nan")
    po = sum(a == b for a, b in pairs) / n
    ca = Counter(a for a, _ in pairs)
    cb = Counter(b for _, b in pairs)
    pe = sum(ca[k] * cb[k] for k in set(ca) | set(cb)) / (n * n)
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)


def mcnemar_exact(b: int, c: int) -> float:
    """Two-sided exact McNemar p-value. b = only A solved, c = only B solved."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def _pct(x: float) -> str:
    return f"{100 * x:.1f}%"


def summarize(runs: list[Run]) -> str:
    by_model: dict[str, list[Run]] = defaultdict(list)
    for r in runs:
        by_model[r.model or "?"].append(r)
    out = []
    if any(r.synthetic for r in runs):
        out.append("> WARNING: includes synthetic example runs. Do not report these numbers.\n")

    out.append("### Resolution rate (RQ4 baseline)\n")
    out.append("| Model | Runs (scored) | Resolved | Rate | 95% CI |")
    out.append("| --- | --- | --- | --- | --- |")
    for m, rs in sorted(by_model.items()):
        scored = [r for r in rs if group_of(r.primary_failure or "") != "excluded"]
        k = sum(r.primary_failure == "RESOLVED" for r in scored)
        lo, hi = wilson(k, len(scored))
        rate = k / len(scored) if scored else 0
        out.append(f"| {m} | {len(scored)} | {k} | {_pct(rate)} | {_pct(lo)}–{_pct(hi)} |")

    out.append("\n### Primary failure cause among failed runs (RQ2)\n")
    labels = sorted({r.primary_failure for r in runs if r.primary_failure})
    out.append("| Model | " + " | ".join(labels) + " | interface share |")
    out.append("| --- | " + " | ".join("---" for _ in labels) + " | --- |")
    for m, rs in sorted(by_model.items()):
        failed = [r for r in rs if group_of(r.primary_failure or "") in ("interface", "budget", "reasoning")]
        c = Counter(r.primary_failure for r in rs)
        inter = sum(group_of(r.primary_failure) == "interface" for r in failed)
        lo, hi = wilson(inter, len(failed))
        share = f"{_pct(inter / len(failed))} ({_pct(lo)}–{_pct(hi)})" if failed else "–"
        out.append(f"| {m} | " + " | ".join(str(c.get(l, 0)) for l in labels) + f" | {share} |")

    out.append("\n### Tool-call problems per 100 turns (RQ1)\n")
    codes = sorted({i for r in runs for s in r.steps for i in s.call_issues})
    mid = (" | ".join(codes) + " | ") if codes else ""
    out.append("| Model | Turns | " + mid + "any problem |")
    out.append("| --- | --- | " + ("".join("--- | " for _ in codes)) + "--- |")
    for m, rs in sorted(by_model.items()):
        steps = [s for r in rs for s in r.steps]
        n = len(steps) or 1
        c = Counter(i for s in steps for i in s.call_issues)
        anyp = sum(bool(s.call_issues) for s in steps)
        vals = "".join(f"{100 * c.get(k, 0) / n:.1f} | " for k in codes)
        out.append(f"| {m} | {len(steps)} | " + vals + f"{100 * anyp / n:.1f} |")
    return "\n".join(out)


def replay(runs: list[Run]) -> str:
    """RQ3 on real data: what the guard would have done to each original turn."""
    status = Counter(s.call_status for r in runs for s in r.steps)
    total = sum(status.values()) or 1
    lines = ["| Guard verdict on original turn | Turns | Share |", "| --- | --- | --- |"]
    for k in ("valid", "repaired", "rejected", "no_call"):
        lines.append(f"| {k} | {status.get(k, 0)} | {_pct(status.get(k, 0) / total)} |")
    broken = status.get("repaired", 0) + status.get("rejected", 0)
    if broken:
        lines.append(f"\nRepairable share of broken calls: {_pct(status.get('repaired', 0) / broken)} "
                     f"({status.get('repaired', 0)}/{broken}). No-call turns are not repairable.")
    return "\n".join(lines)


def agreement(runs: list[Run]) -> str:
    pairs = [(r.primary_failure, r.human_label) for r in runs if r.human_label and r.primary_failure]
    if not pairs:
        return "No hand labels yet (set `human_label` on runs you check)."
    k = cohen_kappa(pairs)
    acc = sum(a == b for a, b in pairs) / len(pairs)
    confusions = Counter((a, b) for a, b in pairs if a != b).most_common(5)
    lines = [f"Hand-labeled runs: {len(pairs)}  |  raw agreement {_pct(acc)}  |  Cohen's kappa {k:.3f}"]
    if confusions:
        lines.append("Top disagreements (auto -> human): " + ", ".join(f"{a}->{b} x{n}" for (a, b), n in confusions))
    return "\n".join(lines)


def compare(a: list[Run], b: list[Run], name_a: str = "A", name_b: str = "B") -> str:
    """Paired before/after on shared task ids (RQ4)."""
    ra = {r.task_id: r.primary_failure == "RESOLVED" for r in a if group_of(r.primary_failure or "") != "excluded"}
    rb = {r.task_id: r.primary_failure == "RESOLVED" for r in b if group_of(r.primary_failure or "") != "excluded"}
    shared = sorted(set(ra) & set(rb))
    n = len(shared)
    ka = sum(ra[t] for t in shared)
    kb = sum(rb[t] for t in shared)
    only_a = sum(ra[t] and not rb[t] for t in shared)
    only_b = sum(rb[t] and not ra[t] for t in shared)
    p = mcnemar_exact(only_a, only_b)
    la, ha = wilson(ka, n)
    lb, hb = wilson(kb, n)
    return (f"| Config | Resolved / {n} shared tasks | 95% CI |\n| --- | --- | --- |\n"
            f"| {name_a} | {ka} ({_pct(ka / n if n else 0)}) | {_pct(la)}–{_pct(ha)} |\n"
            f"| {name_b} | {kb} ({_pct(kb / n if n else 0)}) | {_pct(lb)}–{_pct(hb)} |\n\n"
            f"Only {name_a} solved: {only_a}; only {name_b} solved: {only_b}; exact McNemar p = {p:.4f}")


In [ ]:
%%writefile faultlines/src/faultlines/atlas/taxonomy.py
"""FaultLines failure taxonomy (v0 — freeze as v1 at the end of week 2).

Every run gets exactly ONE primary label, chosen by the precedence rule in
autolabel.py, plus any number of secondary flags. Groups answer RQ2
("interface vs reasoning").
"""

from __future__ import annotations

from enum import Enum


class Primary(str, Enum):
    RESOLVED = "RESOLVED"
    ENV_ERROR = "ENV_ERROR"              # harness / infra failure; excluded from rates
    # --- process failures: the run never produced a considered answer
    INTERFACE = "INTERFACE"              # tool calls kept breaking (truncated, malformed, no call)
    LOOP = "LOOP"                        # repeated the same action(s) without progress
    CONTEXT = "CONTEXT"                  # ran into the context limit / lost state after compaction
    BUDGET = "BUDGET"                    # out of time/turns/calls while otherwise working normally
    # --- solution failures: the run submitted, but the answer was wrong
    NO_PATCH = "NO_PATCH"                # submitted an empty diff
    BROKEN_PATCH = "BROKEN_PATCH"        # patch did not apply, or tests failed to even collect/import
    WRONG_LOCATION = "WRONG_LOCATION"    # edited none of the files the reference fix touches
    WRONG_FIX = "WRONG_FIX"              # edited the right file(s) but tests still fail
    UNRESOLVED_OTHER = "UNRESOLVED_OTHER"  # failed, but no gold files to tell location from fix
    UNKNOWN_OUTCOME = "UNKNOWN_OUTCOME"  # resolution result missing


GROUPS = {
    "interface": {Primary.INTERFACE, Primary.LOOP, Primary.CONTEXT},
    "budget": {Primary.BUDGET},
    "reasoning": {Primary.NO_PATCH, Primary.BROKEN_PATCH, Primary.WRONG_LOCATION,
                  Primary.WRONG_FIX, Primary.UNRESOLVED_OTHER},
    "success": {Primary.RESOLVED},
    "excluded": {Primary.ENV_ERROR, Primary.UNKNOWN_OUTCOME},
}


def group_of(label: str) -> str:
    for g, members in GROUPS.items():
        if label in {m.value for m in members}:
            return g
    return "unknown"


DEFINITIONS = {
    Primary.RESOLVED: "Hidden tests pass (harness Phase 2 exit code 0).",
    Primary.ENV_ERROR: "Run ended by an infrastructure error, not by the agent.",
    Primary.INTERFACE: "Did not submit; >=3 of the last 5 turns had a rejected or missing tool call, "
                       "or the last turn was a truncated call.",
    Primary.LOOP: "Did not submit; a repeat/cycle loop was detected in the final third of the run.",
    Primary.CONTEXT: "Did not submit; prompt size reached >=90% of the context limit, "
                     "or a compaction event occurred.",
    Primary.BUDGET: "Did not submit and none of the above: ran out of time, turns or tool calls.",
    Primary.NO_PATCH: "Submitted with an empty diff.",
    Primary.BROKEN_PATCH: "Patch failed to apply, or the test run failed at collection/import.",
    Primary.WRONG_LOCATION: "Patch touches none of the files in the reference patch.",
    Primary.WRONG_FIX: "Patch touches at least one reference file; tests still fail.",
    Primary.UNRESOLVED_OTHER: "Failed after submitting; no reference files available.",
    Primary.UNKNOWN_OUTCOME: "No resolution result recorded.",
}

SECONDARY_FLAGS = {
    "HIGH_INVALID_RATE": ">=20% of turns had a rejected or missing tool call",
    "HAD_TRUNCATION": "at least one truncated tool call",
    "HAD_LOOP": "loop detected anywhere in the run",
    "CONTEXT_PRESSURE": "prompt reached >=90% of context limit or compaction happened",
    "RUNTIME_ERRORS": ">=3 tool results were errors (edit no-match, file not found, timeout)",
    "USED_CODE_GRAPH": "called a code-intelligence tool at least once",
    "NUDGED": "harness injected a continuation nudge at least once",
}


In [ ]:
%%writefile faultlines/src/faultlines/atlas/trace.py
"""The FaultLines trace record: one agent run on one task.

This is the released dataset's schema. Adapters (adapters.py) convert harness
output into it; the labeler (autolabel.py) fills in the label fields.
"""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator

SCHEMA_VERSION = "faultlines-trace-v0.1"


@dataclass
class Step:
    index: int
    raw_output: str = ""                        # the model's full text for this turn (may include thinking)
    tool_name: str | None = None                # as executed by the harness (None if nothing ran)
    tool_args: dict[str, Any] | None = None
    tool_result: str = ""                       # observation text returned to the model
    prompt_tokens: int | None = None
    completion_tokens: int | None = None
    finish_reason: str | None = None            # e.g. "stop", "length"/"MAX_TOKENS"
    events: list[str] = field(default_factory=list)   # harness events, e.g. "nudge", "compaction"
    extra_calls: list[dict] = field(default_factory=list)  # further calls in the same turn (name, args, result)
    # filled by the labeler
    call_status: str | None = None              # valid | repaired | rejected | no_call
    call_issues: list[str] = field(default_factory=list)
    runtime_error: str | None = None            # e.g. EDIT_NO_MATCH, FILE_NOT_FOUND, TIMEOUT


@dataclass
class Run:
    task_id: str
    model: str = ""
    config_hash: str = ""
    seed: int | None = None
    resolved: bool | None = None
    exit_reason: str = ""                       # submitted | timeout | max_turns | max_tool_calls | error | ...
    agent_patch: str = ""
    patch_applied: bool | None = None
    test_output: str = ""
    gold_files: list[str] = field(default_factory=list)
    context_limit: int | None = None
    steps: list[Step] = field(default_factory=list)
    synthetic: bool = False                     # True for hand-made examples; never mix into results
    harness: str = ""                           # e.g. "faultlines-mini-0.1/strict"
    timing: dict[str, float] = field(default_factory=dict)   # setup_s, agent_s, verify_s
    notes: list[str] = field(default_factory=list)            # environment notes (e.g. online_pip_fallback)
    # filled by the labeler / annotator
    primary_failure: str | None = None
    secondary: list[str] = field(default_factory=list)
    evidence: list[str] = field(default_factory=list)
    human_label: str | None = None
    schema_version: str = SCHEMA_VERSION

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "Run":
        d = dict(d)
        steps = [Step(**s) for s in d.pop("steps", [])]
        known = {f for f in cls.__dataclass_fields__}
        return cls(steps=steps, **{k: v for k, v in d.items() if k in known})

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def read_runs(path: str | Path) -> Iterator[Run]:
    """Read .jsonl (one run per line), a .json list, a single .json run, or a directory of them."""
    p = Path(path)
    if p.is_dir():
        for f in sorted(p.iterdir()):
            if f.suffix in (".json", ".jsonl"):
                yield from read_runs(f)
        return
    text = p.read_text()
    if p.suffix == ".jsonl":
        for line in text.splitlines():
            if line.strip():
                yield Run.from_dict(json.loads(line))
        return
    data = json.loads(text)
    for d in data if isinstance(data, list) else [data]:
        yield Run.from_dict(d)


def write_runs(runs: Iterable[Run], path: str | Path) -> int:
    n = 0
    with open(path, "w") as f:
        for r in runs:
            f.write(json.dumps(r.to_dict()) + "\n")
            n += 1
    return n


In [ ]:
%%writefile faultlines/src/faultlines/cli.py
"""faultlines command-line interface.

    faultlines check  "<model output>"            guard verdict for one turn
    faultlines ingest --results DIR [--tasks tasks.jsonl] --model NAME -o runs.jsonl
    faultlines label  runs.jsonl -o labeled.jsonl  auto-label runs
    faultlines summarize labeled.jsonl             RQ1/RQ2 tables
    faultlines replay labeled.jsonl                RQ3 on real turns
    faultlines agreement labeled.jsonl             kappa vs hand labels
    faultlines compare before.jsonl after.jsonl    RQ4 paired test
    faultlines stress [--from labeled.jsonl]       RQ3 stress set
    faultlines infer-schemas labeled.jsonl -o tools.json
    faultlines harness --data DIR --endpoint URL --model NAME --tokenizer PATH --out DIR [--limit 5]
"""

from __future__ import annotations

import argparse
import json
import sys

from .atlas import label_run, read_runs, write_runs
from .atlas.report import agreement, compare, replay, summarize
from .stress import build_cases, evaluate, report, sample_calls
from .toolguard import ToolGuard, ToolRegistry, infer_registry, validate_call


def _registry(path: str | None) -> ToolRegistry:
    return ToolRegistry.from_file(path) if path else ToolRegistry.default_harness()


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(prog="faultlines")
    ap.add_argument("--tools", help="tool declarations JSON (default: built-in harness guess)")
    sub = ap.add_subparsers(dest="cmd", required=True)

    p = sub.add_parser("check"); p.add_argument("text", nargs="?"); p.add_argument("--file")
    p.add_argument("--lossy", action="store_true")
    p = sub.add_parser("ingest"); p.add_argument("--results", required=True); p.add_argument("--tasks")
    p.add_argument("--model", default=""); p.add_argument("--config-hash", default="")
    p.add_argument("--context-limit", type=int, default=32768); p.add_argument("-o", "--out", required=True)
    p = sub.add_parser("label"); p.add_argument("runs"); p.add_argument("-o", "--out", required=True)
    for name in ("summarize", "replay", "agreement"):
        p = sub.add_parser(name); p.add_argument("runs")
    p = sub.add_parser("compare"); p.add_argument("before"); p.add_argument("after")
    p = sub.add_parser("stress"); p.add_argument("--from", dest="src"); p.add_argument("--seed", type=int, default=0)
    p.add_argument("--lossy", action="store_true")
    p = sub.add_parser("infer-schemas"); p.add_argument("runs"); p.add_argument("-o", "--out", required=True)
    p = sub.add_parser("harness", help="run tasks with the mini-harness against a vLLM endpoint")
    p.add_argument("--data", required=True, help="competition dataset root (has tasks.jsonl, snapshots/)")
    p.add_argument("--endpoint", help="e.g. http://127.0.0.1:8000")
    p.add_argument("--model", help="served model name")
    p.add_argument("--tokenizer", help="path or HF id for the chat template")
    p.add_argument("--gold", action="store_true",
                   help="no model: replay each task's reference patch (environment check; should resolve ~100%%)")
    p.add_argument("--out", required=True)
    p.add_argument("--task-ids", nargs="*"); p.add_argument("--limit", type=int)
    p.add_argument("--mode", choices=["strict", "guard"], default="strict")
    p.add_argument("--max-tool-calls", type=int, default=50); p.add_argument("--time-minutes", type=float, default=30)
    p.add_argument("--max-output-tokens", type=int, default=16384); p.add_argument("--max-model-len", type=int, default=32768)
    p.add_argument("--temperature", type=float, default=0.0); p.add_argument("--seed", type=int, default=0)
    p.add_argument("--thinking", choices=["on", "off", "default"], default="default")
    p.add_argument("--system-prompt", help="file with the system instruction (e.g. sample_submission prompt)")
    p.add_argument("--python", default="python3", help="interpreter for task venvs (e.g. a Python 3.12)")
    p.add_argument("--online-fallback", action="store_true", help="allow online pip if offline install fails")
    p.add_argument("--tool-style", choices=["gemma_inline", "tool_role"], default="gemma_inline")
    p.add_argument("--label", default="")

    a = ap.parse_args(argv)
    reg = _registry(a.tools)

    if a.cmd == "check":
        text = open(a.file).read() if a.file else (a.text if a.text is not None else sys.stdin.read())
        print(json.dumps(ToolGuard(reg, allow_lossy=a.lossy).check(text).to_dict(), indent=2))
    elif a.cmd == "ingest":
        from .atlas.adapters import load_swegemma_results
        n = write_runs(load_swegemma_results(a.results, a.tasks, a.model, a.config_hash, a.context_limit), a.out)
        print(f"wrote {n} runs to {a.out}")
    elif a.cmd == "label":
        n = write_runs((label_run(r, reg) for r in read_runs(a.runs)), a.out)
        print(f"labeled {n} runs -> {a.out}")
    elif a.cmd == "summarize":
        print(summarize(list(read_runs(a.runs))))
    elif a.cmd == "replay":
        print(replay(list(read_runs(a.runs))))
    elif a.cmd == "agreement":
        print(agreement(list(read_runs(a.runs))))
    elif a.cmd == "compare":
        print(compare(list(read_runs(a.before)), list(read_runs(a.after)), "before", "after"))
    elif a.cmd == "stress":
        if a.src:
            calls = [(s.tool_name, s.tool_args or {}) for r in read_runs(a.src) if not r.synthetic
                     for s in r.steps if s.tool_name and not validate_call(s.tool_name, s.tool_args or {}, reg)]
        else:
            calls = sample_calls()
            print("> using built-in sample calls; pass --from labeled.jsonl for real ones\n")
        print(report(evaluate(build_cases(calls, reg, a.seed), reg, a.lossy)))
    elif a.cmd == "infer-schemas":
        calls = [(s.tool_name, s.tool_args or {}) for r in read_runs(a.runs)
                 for s in r.steps if s.tool_name and not (s.runtime_error or "").startswith("TOOL_ERROR")]
        infer_registry(calls).dump(a.out)
        print(f"wrote inferred declarations for {len({c[0] for c in calls})} tools to {a.out}")
    elif a.cmd == "harness":
        return _harness(a)
    return 0


def _gold_model(task):
    from .harness import ScriptedModel
    from .toolguard import render_call
    cmd = f"cd /workspace && git apply <<'FL_GOLD_EOF'\n{task.patch}\nFL_GOLD_EOF"
    return ScriptedModel([render_call("run_command", {"command": cmd}), render_call("submit_patch", {})],
                         name="gold-replay")


def _harness(a) -> int:
    import time
    from pathlib import Path
    from .harness import CompletionEndpointModel, DataLayout, Limits, RunConfig, load_tasks, run_task
    from .harness.prompt import DEFAULT_SYSTEM
    layout = DataLayout.find(a.data)
    tasks = load_tasks(layout.tasks, a.task_ids, a.limit)
    kwargs = {} if a.thinking == "default" else {"enable_thinking": a.thinking == "on"}
    if a.gold:
        model = None
    else:
        if not (a.endpoint and a.model and a.tokenizer):
            raise SystemExit("--endpoint, --model and --tokenizer are required unless --gold is set")
        model = CompletionEndpointModel(a.endpoint, a.model, a.tokenizer, a.max_output_tokens, a.max_model_len,
                                        a.temperature, seed=a.seed, chat_template_kwargs=kwargs,
                                        tool_message_style=a.tool_style)
    cfg = RunConfig(limits=Limits(time_minutes=a.time_minutes, tool_calls=a.max_tool_calls), mode=a.mode,
                    system_prompt=Path(a.system_prompt).read_text() if a.system_prompt else DEFAULT_SYSTEM,
                    python=a.python, allow_online_fallback=a.online_fallback, context_limit=a.max_model_len,
                    seed=a.seed, label=a.label)
    out = Path(a.out); (out / "patches").mkdir(parents=True, exist_ok=True)
    print(f"{len(tasks)} tasks | mode={a.mode} | config={cfg.hash()} | out={out}", flush=True)
    for i, t in enumerate(tasks, 1):
        t0 = time.time()
        m = model if model is not None else _gold_model(t)
        r = run_task(t, layout, m, cfg)
        with open(out / "runs.jsonl", "a") as f:
            f.write(json.dumps(r.to_dict()) + "\n")
        (out / "patches" / f"{t.instance_id}.patch").write_text(r.agent_patch or "")
        print(f"[{i}/{len(tasks)}] {t.instance_id}: resolved={r.resolved} exit={r.exit_reason} turns={len(r.steps)} "
              f"timing={r.timing} wall={time.time() - t0:.0f}s notes={r.notes}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile faultlines/src/faultlines/harness/__init__.py
"""Mini-harness: a faithful local re-implementation of the (unreleased) swegemma evaluator,
built from the competition's published HARNESS_README.md, that records raw model output."""

from .model import CompletionEndpointModel, ModelTurn, ScriptedModel
from .runner import HARNESS_VERSION, RunConfig, run_task
from .tasks import DataLayout, Task, load_tasks
from .tools import Limits

__all__ = ["CompletionEndpointModel", "ModelTurn", "ScriptedModel", "HARNESS_VERSION", "RunConfig", "run_task",
           "DataLayout", "Task", "load_tasks", "Limits"]


In [ ]:
%%writefile faultlines/src/faultlines/harness/editing.py
"""3-tier replacement used by edit_file, as documented for adk-eval-core's apply_replacement.

    exact     character-for-character match (after \r\n -> \n)
    flexible  line-by-line match ignoring leading/trailing whitespace; new text is
              re-indented to the matched block's indentation
    regex     tokens split around code delimiters ( ) : [ ] { } > = < joined by \s*
"""

from __future__ import annotations

import re
from dataclasses import dataclass


class EditError(Exception):
    pass


@dataclass
class EditResult:
    content: str
    occurrences: int
    strategy: str


def _indent(line: str) -> str:
    return line[: len(line) - len(line.lstrip())]


def _exact(content: str, old: str, new: str, allow_multiple: bool) -> EditResult | None:
    n = content.count(old)
    if n == 0:
        return None
    if n > 1 and not allow_multiple:
        raise EditError(f"old_string matches {n} locations; make it unique or set allow_multiple=true")
    return EditResult(content.replace(old, new) if allow_multiple else content.replace(old, new, 1), n, "exact")


def _flexible(content: str, old: str, new: str, allow_multiple: bool) -> EditResult | None:
    lines = content.split("\n")
    old_lines = old.strip("\n").split("\n")
    target = [l.strip() for l in old_lines]
    if not any(target):
        return None
    k = len(target)
    hits = [i for i in range(len(lines) - k + 1) if [l.strip() for l in lines[i:i + k]] == target]
    if not hits:
        return None
    if len(hits) > 1 and not allow_multiple:
        raise EditError(f"old_string matches {len(hits)} locations (whitespace-insensitive); make it unique")
    new_lines = new.strip("\n").split("\n")
    old_base = _indent(next(l for l in old_lines if l.strip()))
    for i in reversed(hits):
        base = _indent(next(l for l in lines[i:i + k] if l.strip()))
        reindented = []
        for l in new_lines:
            if not l.strip():
                reindented.append("")
            elif l.startswith(old_base):
                reindented.append(base + l[len(old_base):])
            else:
                reindented.append(base + l.lstrip())
        lines[i:i + k] = reindented
    return EditResult("\n".join(lines), len(hits), "flexible")


_DELIMS = r"([()\[\]{}:<>=,])"


def _regex(content: str, old: str, new: str, allow_multiple: bool) -> EditResult | None:
    tokens = [t for t in re.split(r"\s+|" + _DELIMS, old) if t]
    if not tokens:
        return None
    pattern = r"\s*".join(re.escape(t) for t in tokens)
    matches = list(re.finditer(pattern, content))
    if not matches:
        return None
    if len(matches) > 1 and not allow_multiple:
        raise EditError(f"old_string matches {len(matches)} locations (token match); make it unique")
    out = content
    for m in reversed(matches if allow_multiple else matches[:1]):
        out = out[: m.start()] + new + out[m.end():]
    return EditResult(out, len(matches), "regex")


def apply_replacement(content: str, old: str, new: str, allow_multiple: bool = False) -> EditResult:
    if old == "":
        raise EditError("old_string must not be empty")
    content = content.replace("\r\n", "\n")
    old = old.replace("\r\n", "\n")
    new = new.replace("\r\n", "\n")
    for strategy in (_exact, _flexible, _regex):
        res = strategy(content, old, new, allow_multiple)
        if res is not None:
            return res
    raise EditError("old_string not found in file")


In [ ]:
%%writefile faultlines/src/faultlines/harness/graph.py
"""Code-intelligence tools over the competition's graphs/ and embeddings/ files.

Graph: NetworkX node-link JSON (nodes[*].id/name/text, edges[*].source/target/type).
Embeddings: .npz with one 256-d float32 vector per node id.
search_similar_code resolves the query to a node key (no live embedding server,
as in the official harness) and ranks all nodes by cosine similarity.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any


class CodeGraph:
    def __init__(self, graph_path: Path | None, emb_path: Path | None):
        self.nodes: dict[str, dict] = {}
        self.out_edges: dict[str, list[tuple[str, str]]] = {}
        self.in_edges: dict[str, list[tuple[str, str]]] = {}
        self.vectors = None
        self.vec_keys: list[str] = []
        if graph_path and Path(graph_path).exists() and Path(graph_path).stat().st_size > 100:
            g = json.loads(Path(graph_path).read_text())
            for n in g.get("nodes", []):
                self.nodes[str(n["id"])] = n
            for e in g.get("edges", g.get("links", [])):
                s, t, ty = str(e["source"]), str(e["target"]), str(e.get("type", "RELATED"))
                self.out_edges.setdefault(s, []).append((t, ty))
                self.in_edges.setdefault(t, []).append((s, ty))
        if emb_path and Path(emb_path).exists() and Path(emb_path).stat().st_size > 100:
            import numpy as np
            z = np.load(emb_path)
            self.vec_keys = [k[:-4] if k.endswith(".npy") else k for k in z.files]
            mat = np.stack([z[k] for k in z.files]).astype("float32")
            norms = np.linalg.norm(mat, axis=1, keepdims=True)
            self.vectors = mat / np.maximum(norms, 1e-8)

    @property
    def available(self) -> bool:
        return bool(self.nodes) or self.vectors is not None

    # 4-tier resolution: exact -> suffix (after . or /) -> case-insensitive -> substring
    @staticmethod
    def _resolve(name: str, keys: list[str]) -> str | None:
        if not name:
            return None
        if name in keys:
            return name
        suf = [k for k in keys if k.endswith("." + name) or k.endswith("/" + name)]
        if suf:
            return min(suf, key=len)
        low = name.lower()
        ci = [k for k in keys if k.lower() == low]
        if ci:
            return ci[0]
        sub = [k for k in keys if low in k.lower()]
        return min(sub, key=len) if sub else None

    def resolve(self, name: str) -> str | None:
        return self._resolve(name, list(self.nodes))

    def neighbors(self, node: str, edge_type: str | None = None, max_neighbors: int = 50) -> dict[str, Any]:
        rid = self.resolve(node)
        if rid is None:
            return {"status": "error", "error_type": "NodeNotFound", "error_message": f"No graph node matches '{node}'"}
        et = edge_type.lower() if edge_type else None
        items = [{"node": t, "direction": "out", "type": ty} for t, ty in self.out_edges.get(rid, [])]
        items += [{"node": s, "direction": "in", "type": ty} for s, ty in self.in_edges.get(rid, [])]
        if et:
            items = [i for i in items if i["type"].lower() == et]
        items = items[: max(1, int(max_neighbors))]
        return {"status": "ok", "node": rid, "neighbors": items, "count": len(items)}

    def similar(self, query: str, k: int = 10) -> dict[str, Any]:
        if self.vectors is None:
            return {"status": "error", "error_type": "NoEmbeddings", "error_message": "No embeddings for this repo"}
        key = self._resolve(query, self.vec_keys)
        if key is None:
            return {"status": "error", "error_type": "NodeNotFound",
                    "error_message": f"'{query}' matches no embedded symbol; pass a class/function name"}
        i = self.vec_keys.index(key)
        sims = self.vectors @ self.vectors[i]
        order = [j for j in sims.argsort()[::-1] if j != i][: max(1, int(k))]
        res = [{"node_name": self.vec_keys[j], "code": str(self.nodes.get(self.vec_keys[j], {}).get("text", ""))[:1500],
                "similarity": round(float(sims[j]), 4)} for j in order]
        return {"status": "ok", "query": key, "results": res, "count": len(res)}

    def subgraph(self, nodes: list[str]) -> dict[str, Any]:
        ids = [r for r in (self.resolve(str(n)) for n in (nodes or [])) if r]
        idset = set(ids)
        edges = [{"from": s, "to": t, "type": ty} for s in ids for t, ty in self.out_edges.get(s, []) if t in idset]
        return {"status": "ok", "nodes": ids, "edges": edges, "node_count": len(ids), "edge_count": len(edges)}


In [ ]:
%%writefile faultlines/src/faultlines/harness/model.py
"""Model clients.

CompletionEndpointModel renders the Gemma 4 chat template locally (transformers
tokenizer) and calls an OpenAI-compatible /v1/completions endpoint (vLLM) with
special tokens kept. That way the *raw* model text, including broken
<|tool_call> spans, is recorded, which the official harness never shows.

ScriptedModel replays fixed outputs for tests and demos.
"""

from __future__ import annotations

import copy
import json
import urllib.request
from dataclasses import dataclass
from typing import Any, Protocol


@dataclass
class ModelTurn:
    text: str
    finish_reason: str | None = None
    prompt_tokens: int | None = None
    completion_tokens: int | None = None


class ChatModel(Protocol):
    name: str

    def generate(self, messages: list[dict], tools: list[dict]) -> ModelTurn: ...


class ScriptedModel:
    def __init__(self, outputs: list[str | ModelTurn], name: str = "scripted"):
        self.outputs = list(outputs)
        self.name = name
        self.calls = 0

    def generate(self, messages, tools) -> ModelTurn:
        self.calls += 1
        if not self.outputs:
            return ModelTurn("I have nothing more to do.", "stop", 0, 0)
        o = self.outputs.pop(0)
        return o if isinstance(o, ModelTurn) else ModelTurn(o, "stop", 1000 * self.calls, 50)


def to_gemma_inline(messages: list[dict]) -> list[dict]:
    """Fold role=tool messages into the preceding assistant message as `tool_responses`
    (the shape shown in Google's Gemma 4 function-calling docs)."""
    out: list[dict] = []
    for m in messages:
        if m.get("role") == "tool" and out and out[-1].get("role") == "assistant":
            try:
                resp = json.loads(m["content"])
            except (json.JSONDecodeError, TypeError):
                resp = {"result": m["content"]}
            out[-1].setdefault("tool_responses", []).append({"name": m.get("name", ""), "response": resp})
        else:
            out.append(copy.deepcopy(m))
    return out


class CompletionEndpointModel:
    def __init__(self, base_url: str, model: str, tokenizer_path: str, max_output_tokens: int = 16384,
                 max_model_len: int = 32768, temperature: float = 0.0, top_p: float | None = None,
                 seed: int | None = 0, chat_template_kwargs: dict | None = None,
                 tool_message_style: str = "gemma_inline", timeout: int = 1800):
        from transformers import AutoTokenizer
        self.base_url = base_url.rstrip("/")
        self.name = model
        self.tok = AutoTokenizer.from_pretrained(tokenizer_path)
        self.max_output_tokens = max_output_tokens
        self.max_model_len = max_model_len
        self.temperature, self.top_p, self.seed = temperature, top_p, seed
        self.template_kwargs = chat_template_kwargs or {}
        self.style = tool_message_style
        self.timeout = timeout

    def render(self, messages: list[dict], tools: list[dict]) -> str:
        msgs = to_gemma_inline(messages) if self.style == "gemma_inline" else messages
        return self.tok.apply_chat_template(msgs, tools=[{"type": "function", "function": t} for t in tools],
                                            add_generation_prompt=True, tokenize=False, **self.template_kwargs)

    def count(self, text: str) -> int:
        return len(self.tok(text, add_special_tokens=False)["input_ids"])

    def generate(self, messages: list[dict], tools: list[dict]) -> ModelTurn:
        prompt = self.render(messages, tools)
        n = self.count(prompt)
        max_tokens = min(self.max_output_tokens, self.max_model_len - n - 8)
        if max_tokens < 64:
            return ModelTurn("", "context_overflow", n, 0)
        body: dict[str, Any] = {"model": self.name, "prompt": prompt, "max_tokens": max_tokens,
                                "temperature": self.temperature, "skip_special_tokens": False,
                                "spaces_between_special_tokens": False, "add_special_tokens": False}
        if self.top_p is not None:
            body["top_p"] = self.top_p
        if self.seed is not None:
            body["seed"] = self.seed
        req = urllib.request.Request(f"{self.base_url}/v1/completions", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=self.timeout) as r:
            data = json.loads(r.read())
        ch = data["choices"][0]
        usage = data.get("usage") or {}
        return ModelTurn(ch.get("text", ""), ch.get("finish_reason"), usage.get("prompt_tokens", n),
                         usage.get("completion_tokens"))


In [ ]:
%%writefile faultlines/src/faultlines/harness/prompt.py
"""Task prompt and continuation nudges, following HARNESS_README.md section 5.

Sections 1-5 and 6-7 follow the published text. The "Standard Instructions (0-5)"
wording is not published verbatim, so it is paraphrased from the README's summary.
The three nudge messages are copied from the README's table.
"""

from __future__ import annotations

from .tasks import Task
from .tools import Limits

NUDGE_UNCLOSED_CALL = (
    "Your previous response reached the token limit before the tool call finished closing (<|tool_call|> was "
    "cut off). Do NOT repeat your prior reasoning in thought—emit your next tool call immediately, and if calling "
    "edit_file or write_file, split the change into smaller incremental edits.")
NUDGE_MAX_TOKENS = (
    "Your previous response reached the token limit while thinking before a tool call was completed. Do NOT repeat "
    "your analysis in thought—keep reasoning under a few sentences and emit your next tool call immediately, or call "
    "submit_patch when you have completed and verified your changes.")
NUDGE_CONTINUE = ("Please continue your work using the available tools, or call submit_patch when you have completed "
                  "and verified your changes.")

DEFAULT_SYSTEM = """You are an autonomous software engineer fixing an issue in a Python repository at /workspace.
Explore the code with the tools, find the root cause, make a minimal fix in the library code (not the tests),
check it with a quick targeted command, then call submit_patch as your final action.
Always act through tool calls; keep reasoning short."""


def build_agent_prompt(task: Task, limits: Limits, has_graph: bool, layout: str) -> str:
    parts = [f"You are evaluating a software engineering task for repository {task.repo}.\n\n"
             f"Problem Statement:\n{task.problem_statement}"]
    if task.hints_text and task.hints_text.strip():
        parts.append(f"## Hints:\n{task.hints_text.strip()}")
    budget = ["## Task Budget (Session terminates when any budget is exhausted)",
              f"- Time allowance: {limits.time_minutes:.1f} minutes"]
    if limits.tool_calls is not None:
        budget.append(f"- Tool calls allowance: {limits.tool_calls} calls")
    budget.append(f"- Max loop iterations: {limits.turns} turns")
    parts.append("\n".join(budget))
    parts.append("\n".join([
        "## Execution Environment Rules",
        f"- Single command timeout: {limits.command_timeout} seconds (commands exceeding this fail without ending the session)",
        f"- Command output limit: {limits.max_stdout_chars} characters",
        f"- File view limit: {limits.max_file_lines} lines per read_file call",
        f"- File character limit: {limits.max_file_chars} characters per read_file call",
        "- Environment is offline (no network/PyPI access). All repository and test dependencies are ALREADY "
        "pre-installed. Do NOT attempt to run pip install or download packages.",
    ]))
    parts.append("\n".join([
        "## Instructions",
        "0. Work strictly inside /workspace.",
        "1. Inspect the relevant code and follow the repository's existing conventions.",
        "2. Make the smallest change in the library code that resolves the issue; do not edit the tests.",
        "3. Prefer quick inline checks (python3 -c \"...\" assertions) over full test sweeps.",
        "4. Put scratch files in /tmp, never in /workspace, so they do not end up in the patch.",
        "5. When done and verified, call submit_patch, then reply with a short final summary.",
    ]))
    if has_graph:
        parts.append("\n".join([
            "## Code Intelligence Tools",
            "This repository has pre-built code graph and embedding data. Use these tools for fast, targeted navigation:",
            "- `search_similar_code(query)`: Find semantically similar functions/classes by keyword.",
            "- `get_code_neighbors(node)`: Find callers, callees, and definitions related to a symbol.",
            "- `get_code_subgraph(nodes)`: Get the induced subgraph for a set of symbols.",
        ]))
    parts.append("## Workspace Layout\n" + layout)
    return "\n\n".join(parts)


def workspace_layout(sandbox) -> str:
    r = sandbox.run("find . -maxdepth 3 -not -path './.git*' -not -name '__pycache__' -not -name '*.pyc' "
                    "| sort | head -150", timeout=30)
    return r.stdout.strip()


In [ ]:
%%writefile faultlines/src/faultlines/harness/runner.py
"""Agent loop modeled on swegemma's agent_runner (HARNESS_README.md sections 5 and 7).

mode="strict": a tool call counts only if Gemma syntax parses exactly (like the
               harness's vLLM gemma4 parser); anything else is plain text.
mode="guard":  model text goes through toolguard; safe repairs are executed and
               rejected calls get specific feedback instead of a generic nudge.

Output is a FaultLines `Run` with the raw model text for every turn.
"""

from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
import time
import traceback
from dataclasses import asdict, dataclass, field
from pathlib import Path

from ..atlas.autolabel import patch_files
from ..atlas.trace import Run, Step
from ..toolguard import ToolGuard, ToolRegistry, extract_calls, parse_body
from ..toolguard.gemma_syntax import CALL_OPEN, GemmaSyntaxError
from ..toolguard.repair import ToolCall
from .graph import CodeGraph
from .model import ChatModel
from .prompt import (DEFAULT_SYSTEM, NUDGE_CONTINUE, NUDGE_MAX_TOKENS, NUDGE_UNCLOSED_CALL, build_agent_prompt,
                     workspace_layout)
from .sandbox import SubprocessSandbox
from .tasks import DataLayout, Task
from .tools import Limits, ToolContext
from .verify import verify_task

HARNESS_VERSION = "faultlines-mini-0.1"


@dataclass
class RunConfig:
    limits: Limits = field(default_factory=Limits)
    mode: str = "strict"                 # strict | guard
    max_nudges: int = 3
    system_prompt: str = DEFAULT_SYSTEM
    compaction_tokens: int | None = 24000   # compact history when a prompt exceeds this (None = never)
    retention_messages: int = 10            # messages kept after compaction (besides system + task)
    python: str = "python3"
    allow_online_fallback: bool = False
    system_site_packages: bool = False      # tests only
    context_limit: int = 32768
    seed: int | None = 0
    label: str = ""                         # free-text config label for the paper

    def hash(self) -> str:
        d = asdict(self)
        return hashlib.sha256(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]


def strict_calls(text: str) -> list[ToolCall]:
    calls = []
    for raw in extract_calls(text):
        if not raw.closed:
            continue
        try:
            pb = parse_body(raw.body)
        except GemmaSyntaxError:
            continue
        calls.append(ToolCall(pb.name, pb.args))
    return calls


def _visible(text: str) -> str:
    i = text.find(CALL_OPEN)
    return (text if i < 0 else text[:i]).strip()


def _compact(messages: list[dict], keep: int) -> tuple[list[dict], int]:
    head, tail = messages[:2], messages[2:]
    if len(tail) <= keep:
        return messages, 0
    kept = tail[-keep:]
    while kept and kept[0].get("role") == "tool":   # never start on an orphan tool result
        kept = kept[1:]
    removed = len(tail) - len(kept)
    note = {"role": "user", "content": f"[Context compacted: {removed} earlier messages were removed. "
                                       "Re-read files if you need their contents again.]"}
    return head + [note] + kept, removed


def run_task(task: Task, layout: DataLayout, model: ChatModel, cfg: RunConfig, keep_sandbox: bool = False) -> Run:
    run = Run(task_id=task.instance_id, model=model.name, config_hash=cfg.hash(), seed=cfg.seed,
              gold_files=patch_files(task.patch), context_limit=cfg.context_limit,
              harness=f"{HARNESS_VERSION}/{cfg.mode}")
    venv_dir = Path(tempfile.mkdtemp(prefix="flvenv_"))
    sb = SubprocessSandbox(layout.snapshot(task.instance_id), layout.wheels if layout.wheels.exists() else None,
                           python=cfg.python, allow_online_fallback=cfg.allow_online_fallback,
                           venv_path=venv_dir / "venv", keep=keep_sandbox,
                           system_site_packages=cfg.system_site_packages)
    try:
        run.timing["setup_s"] = round(sb.setup(), 1)
        run.notes += sb.notes
        graph = CodeGraph(layout.graph(task.instance_id), layout.embeddings(task.instance_id))
        ctx = ToolContext(sandbox=sb, limits=cfg.limits, graph=graph)
        specs = ToolRegistry.default_harness()
        available = ctx.registry()
        tools = [specs.get(n).to_declaration() for n in available if specs.get(n)]
        guard = ToolGuard(ToolRegistry([specs.get(n) for n in available if specs.get(n)]), detect_loops=False)
        prompt = build_agent_prompt(task, cfg.limits, graph.available, workspace_layout(sb))
        messages: list[dict] = [{"role": "system", "content": cfg.system_prompt}, {"role": "user", "content": prompt}]

        ctx.agent_start = time.time()
        nudges = turns = over_budget_turns = 0
        exit_reason = ""
        while True:
            if ctx.remaining_seconds() <= 0:
                exit_reason = "timeout"; break
            if turns >= cfg.limits.turns:
                exit_reason = "max_turns"; break
            turns += 1
            try:
                t = model.generate(messages, tools)
            except Exception as e:  # server down, OOM, ...
                run.evidence.append(f"model_error: {type(e).__name__}: {str(e)[:300]}")
                exit_reason = "error"; break
            step = Step(index=len(run.steps), raw_output=t.text, prompt_tokens=t.prompt_tokens,
                        completion_tokens=t.completion_tokens, finish_reason=t.finish_reason)
            if t.finish_reason == "context_overflow":
                step.events.append("context_overflow")

            feedback = ""
            if cfg.mode == "guard":
                g = guard.check(t.text)
                calls = g.calls if g.status in ("valid", "repaired") else []
                if g.status == "repaired":
                    step.events.append("guard_repaired:" + ",".join(a for o in g.outcomes for a in o.applied))
                if g.status == "rejected":
                    feedback = g.feedback
                    step.events.append("guard_rejected")
            else:
                calls = strict_calls(t.text)

            if calls:
                nudges = 0
                messages.append({"role": "assistant", "content": _visible(t.text),
                                 "tool_calls": [{"type": "function", "function": {"name": c.name, "arguments": c.args}}
                                                for c in calls]})
                for j, c in enumerate(calls):
                    result = ctx.call(c.name, c.args)
                    messages.append({"role": "tool", "name": c.name, "content": result})
                    if j == 0:
                        step.tool_name, step.tool_args, step.tool_result = c.name, c.args, result[:20000]
                    else:
                        step.extra_calls.append({"name": c.name, "args": c.args, "result": result[:5000]})
                run.steps.append(step)
                if ctx.patch_submitted:
                    exit_reason = "submitted"; break
                rem = ctx.remaining_calls()
                if rem is not None and rem <= 0:
                    over_budget_turns += 1
                    if over_budget_turns > 1:
                        exit_reason = "max_tool_calls"; break
            else:
                nudges += 1
                messages.append({"role": "assistant", "content": _visible(t.text) or "(no tool call)"})
                if nudges > cfg.max_nudges:
                    run.steps.append(step)
                    exit_reason = "max_nudges"; break
                if feedback:
                    msg = feedback
                elif CALL_OPEN in t.text:
                    msg = NUDGE_UNCLOSED_CALL
                elif (t.finish_reason or "").lower() in ("length", "max_tokens"):
                    msg = NUDGE_MAX_TOKENS
                else:
                    msg = NUDGE_CONTINUE
                messages.append({"role": "user", "content": msg})
                step.events.append("nudge")
                run.steps.append(step)

            if cfg.compaction_tokens and (t.prompt_tokens or 0) > cfg.compaction_tokens:
                messages, removed = _compact(messages, cfg.retention_messages)
                if removed:
                    step.events.append(f"compaction:{removed}")

        run.timing["agent_s"] = round(time.time() - ctx.agent_start, 1)
        run.exit_reason = exit_reason
        run.agent_patch = ctx.submitted_patch if ctx.patch_submitted else sb.diff()
    except Exception as e:
        run.exit_reason = "error"
        run.evidence.append("harness_error: " + "".join(traceback.format_exception_only(type(e), e)).strip()[:500])
        return run
    finally:
        sb.cleanup()

    try:
        if run.agent_patch.strip():
            v = verify_task(task, layout, run.agent_patch, venv_dir / "venv", cfg.python, cfg.allow_online_fallback)
            run.resolved, run.patch_applied, run.test_output = v.resolved, v.patch_applied, v.output
            run.timing["verify_s"] = round(v.seconds, 1)
        else:
            run.resolved, run.patch_applied = False, None
    except Exception as e:
        run.evidence.append("verify_error: " + str(e)[:300])
        run.exit_reason = run.exit_reason or "error"
    finally:
        shutil.rmtree(venv_dir, ignore_errors=True)
    return run


In [ ]:
%%writefile faultlines/src/faultlines/harness/sandbox.py
"""Subprocess sandbox modeled on swegemma's `--sandbox subprocess` backend.

Per task: <tmp>/workspace (repo snapshot), <tmp>/tmp, and a venv with the repo
installed in editable mode from the offline wheels. Commands run in bash with
/workspace and /tmp rewritten to the sandbox paths, in their own process group,
killed as a group on timeout.

Documented behaviour reproduced: snapshot extraction, .git/info/exclude
patterns, offline editable install, baseline commit, git add -N . && git diff HEAD.
Not reproduced: Docker isolation, 4 GiB / 2 vCPU limits, the harness's own
pytest.ini/conftest.py (we pass the documented pytest flags on the command line).
"""

from __future__ import annotations

import os
import shutil
import signal
import subprocess
import tarfile
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path

GIT_EXCLUDE = ["__pycache__/", "*.pyc", ".pytest_cache/", "*.egg-info/", "build/", "dist/", ".coverage"]
BUILD_BACKENDS = ["setuptools", "wheel", "hatchling", "hatch-vcs", "flit_core", "pdm-backend", "poetry-core",
                  "setuptools_scm", "pytest"]
GIT = ["git", "-c", "user.email=faultlines@local", "-c", "user.name=faultlines", "-c", "commit.gpgsign=false"]


@dataclass
class CmdResult:
    exit_code: int
    stdout: str
    stderr: str
    timed_out: bool = False
    seconds: float = 0.0


@dataclass
class SubprocessSandbox:
    snapshot: Path
    wheels: Path | None = None
    python: str = "python3"               # interpreter used to build the venv (e.g. a uv-managed 3.12)
    allow_online_fallback: bool = False   # retry pip online if offline install fails (recorded in notes)
    system_site_packages: bool = False    # for tests only
    keep: bool = False
    venv_path: Path | None = None         # where to build the venv (shared by the agent and verify phases)
    root: Path = field(init=False)
    venv: Path | None = None
    notes: list[str] = field(default_factory=list)

    def __post_init__(self):
        self.root = Path(tempfile.mkdtemp(prefix="flsbx_"))
        (self.root / "tmp").mkdir()

    # ------------------------------------------------------------------ paths
    @property
    def workspace(self) -> Path:
        return self.root / "workspace"

    def _env(self) -> dict:
        env = {k: v for k, v in os.environ.items() if not k.startswith("VIRTUAL_ENV")}
        env["TMPDIR"] = str(self.root / "tmp")
        env["TEST_TMPDIR"] = str(self.root / "tmp")
        env["PIP_NO_INPUT"] = "1"
        env["PYTHONDONTWRITEBYTECODE"] = "1"
        env["GIT_TERMINAL_PROMPT"] = "0"
        if self.venv:
            env["VIRTUAL_ENV"] = str(self.venv)
            env["PATH"] = f"{self.venv / 'bin'}:{env.get('PATH', '')}"
        return env

    def rewrite(self, command: str) -> str:
        """Map the agent's absolute paths onto the sandbox, as the official subprocess backend does."""
        # Placeholders first, so a sandbox path that itself lives under /tmp is not rewritten twice.
        s = (command.replace("/workspace", "\x00WS\x00").replace("/tmp/", "\x00TMP\x00")
                    .replace("/wheels", "\x00WH\x00"))
        return (s.replace("\x00WS\x00", str(self.workspace))
                 .replace("\x00TMP\x00", str(self.root / "tmp") + "/")
                 .replace("\x00WH\x00", str(self.wheels or "/wheels")))

    # ---------------------------------------------------------------- running
    def run(self, command: str, timeout: float = 300, cwd: Path | None = None, rewrite: bool = True) -> CmdResult:
        cmd = self.rewrite(command) if rewrite else command
        t0 = time.time()
        proc = subprocess.Popen(["/bin/bash", "-c", cmd], cwd=str(cwd or self.workspace), env=self._env(),
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, start_new_session=True)
        try:
            out, err = proc.communicate(timeout=timeout)
            return CmdResult(proc.returncode, out.decode("utf-8", "replace"), err.decode("utf-8", "replace"),
                             False, time.time() - t0)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            out, err = proc.communicate()
            return CmdResult(-9, out.decode("utf-8", "replace"), err.decode("utf-8", "replace"), True,
                             time.time() - t0)

    def _git(self, *args: str, cwd: Path | None = None, check: bool = True) -> CmdResult:
        p = subprocess.run([*GIT, *args], cwd=str(cwd or self.workspace), capture_output=True, text=True)
        if check and p.returncode != 0:
            raise RuntimeError(f"git {' '.join(args)} failed: {p.stderr[-500:]}")
        return CmdResult(p.returncode, p.stdout, p.stderr)

    # ------------------------------------------------------------------ setup
    def extract(self) -> None:
        if self.workspace.exists():
            shutil.rmtree(self.workspace)
        self.workspace.mkdir()
        with tarfile.open(self.snapshot) as tf:
            try:
                tf.extractall(self.workspace, filter="tar")
            except TypeError:  # Python < 3.12
                tf.extractall(self.workspace)
        # Archives may wrap the repo in one top-level folder: flatten it.
        entries = [p for p in self.workspace.iterdir()]
        if len(entries) == 1 and entries[0].is_dir() and (entries[0] / ".git").exists():
            inner = entries[0]
            for p in inner.iterdir():
                shutil.move(str(p), str(self.workspace / p.name))
            inner.rmdir()
        if not (self.workspace / ".git").exists():
            self._git("init", "-q")
        exclude = self.workspace / ".git" / "info" / "exclude"
        exclude.parent.mkdir(parents=True, exist_ok=True)
        with open(exclude, "a") as f:
            f.write("\n" + "\n".join(GIT_EXCLUDE) + "\n")

    def _pip(self, args: str, online: bool = False) -> CmdResult:
        find = "" if online or not self.wheels else f"--no-index --find-links {self.wheels}"
        return self.run(f"python -m pip install -q {find} {args}", timeout=900, rewrite=False)

    def create_venv(self) -> None:
        self.venv = Path(self.venv_path) if self.venv_path else self.root / "venv"
        flag = "--system-site-packages" if self.system_site_packages else ""
        r = subprocess.run(f"{self.python} -m venv {flag} {self.venv}", shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"venv creation failed: {r.stderr[-500:]}")
        for pkg in BUILD_BACKENDS:  # best effort; not every backend is in the wheels dir
            self._pip(pkg)

    def install_repo(self) -> None:
        """Offline editable install with dependencies, then any requirements*.txt, best effort."""
        r = self._pip("-e .")
        if r.exit_code != 0:
            r = self._pip("--no-build-isolation -e .")
        if r.exit_code != 0 and self.allow_online_fallback:
            self.notes.append("online_pip_fallback")
            r = self._pip("-e .", online=True)
        if r.exit_code != 0:
            self.notes.append("editable_install_failed")
        for req in sorted(self.workspace.glob("requirements*.txt")) + sorted(self.workspace.glob("requirements/*.txt")):
            rr = self._pip(f"-r {req}")
            if rr.exit_code != 0 and self.allow_online_fallback:
                rr = self._pip(f"-r {req}", online=True)
            if rr.exit_code != 0:
                self.notes.append(f"requirements_partial:{req.name}")

    def baseline_commit(self, message: str = "baseline") -> None:
        self._git("add", "-A")
        self._git("commit", "-q", "--allow-empty", "--no-verify", "-m", message)

    def setup(self, reuse_venv: Path | None = None) -> float:
        """Full bootstrap. Returns seconds taken (excluded from the agent's time budget)."""
        t0 = time.time()
        self.extract()
        if reuse_venv is not None and Path(reuse_venv, "bin", "python").exists():
            self.venv = Path(reuse_venv)
        else:
            self.create_venv()
        self.install_repo()
        self.baseline_commit()
        return time.time() - t0

    # ------------------------------------------------------------------ patch
    def diff(self) -> str:
        self._git("add", "-N", ".", check=False)
        return self._git("diff", "HEAD", check=False).stdout

    def cleanup(self) -> None:
        if not self.keep:
            shutil.rmtree(self.root, ignore_errors=True)


In [ ]:
%%writefile faultlines/src/faultlines/harness/tasks.py
"""Competition tasks (tasks.jsonl) and dataset layout.

Dataset (Kaggle: gemma-4-developer-agent, "Data" tab):
    tasks.jsonl                 129 dev tasks (instance_id, repo, base_commit, problem_statement,
                                hints_text, patch, test_patch, created_at)
    snapshots/<id>.tgz          git repo frozen at base_commit, no future history
    graphs/<id>.json            NetworkX node-link call/dependency graph
    embeddings/<id>.npz         256-d vector per graph node
    wheels/                     offline wheels for repo + test dependencies
    sample_submission/          baseline ADK agent config
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Task:
    instance_id: str
    repo: str
    base_commit: str = ""
    problem_statement: str = ""
    hints_text: str = ""
    patch: str = ""
    test_patch: str = ""
    created_at: str = ""

    @classmethod
    def from_dict(cls, d: dict) -> "Task":
        known = cls.__dataclass_fields__
        return cls(**{k: (v if v is not None else "") for k, v in d.items() if k in known})


def load_tasks(path: str | Path, ids: list[str] | None = None, limit: int | None = None) -> list[Task]:
    tasks = []
    for line in Path(path).read_text().splitlines():
        if line.strip():
            tasks.append(Task.from_dict(json.loads(line)))
    if ids:
        wanted = set(ids)
        tasks = [t for t in tasks if t.instance_id in wanted]
    if limit is not None:
        tasks = tasks[:limit]
    return tasks


def patch_targets(patch: str) -> list[str]:
    """Files a unified diff writes to (+++ b/<path>), excluding /dev/null."""
    out = []
    for m in re.finditer(r"^\+\+\+ (?:b/)?(\S+)", patch or "", re.M):
        if m.group(1) != "/dev/null":
            out.append(m.group(1))
    return sorted(set(out))


@dataclass
class DataLayout:
    root: Path

    @classmethod
    def find(cls, root: str | Path) -> "DataLayout":
        """Accept the dataset root or any parent (Kaggle mounts under /kaggle/input/<slug>/)."""
        root = Path(root)
        for cand in [root, *sorted(root.glob("*")), *sorted(root.glob("*/*"))]:
            if (cand / "tasks.jsonl").exists() and (cand / "snapshots").exists():
                return cls(cand)
        raise FileNotFoundError(f"no tasks.jsonl + snapshots/ under {root}")

    @property
    def tasks(self) -> Path:
        return self.root / "tasks.jsonl"

    def snapshot(self, task_id: str) -> Path:
        return self.root / "snapshots" / f"{task_id}.tgz"

    def graph(self, task_id: str) -> Path:
        return self.root / "graphs" / f"{task_id}.json"

    def embeddings(self, task_id: str) -> Path:
        return self.root / "embeddings" / f"{task_id}.npz"

    @property
    def wheels(self) -> Path:
        return self.root / "wheels"

    @property
    def sample_submission(self) -> Path:
        return self.root / "sample_submission"


In [ ]:
%%writefile faultlines/src/faultlines/harness/tools.py
"""The 9 swegemma tools, reproduced from HARNESS_README.md section 6.

Every tool returns a JSON string: {"status": "ok", ...} or
{"status": "error", "error_type": ..., "error_message": ..., "details": {...}}.
get_status returns a raw object and, like submit_patch, does not count as a tool call.
"""

from __future__ import annotations

import difflib
import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

from .editing import EditError, apply_replacement
from .graph import CodeGraph
from .sandbox import SubprocessSandbox


@dataclass
class Limits:
    time_minutes: float = 60.0
    tool_calls: int | None = 50
    turns: int = 500
    command_timeout: int = 300
    max_stdout_chars: int = 5000
    max_file_lines: int = 150
    max_file_chars: int = 10000


def _err(error_type: str, message: str, details: dict | None = None) -> dict:
    d = {"status": "error", "error_type": error_type, "error_message": message}
    if details:
        d["details"] = details
    return d


@dataclass
class ToolContext:
    sandbox: SubprocessSandbox
    limits: Limits
    graph: CodeGraph | None = None
    tool_calls_used: int = 0
    patch_submitted: bool = False
    submitted_patch: str = ""
    agent_start: float = field(default_factory=time.time)

    # ------------------------------------------------------------ budgeting
    def remaining_seconds(self) -> float:
        return self.limits.time_minutes * 60 - (time.time() - self.agent_start)

    def remaining_calls(self) -> int | None:
        return None if self.limits.tool_calls is None else self.limits.tool_calls - self.tool_calls_used

    def _gate(self, count: bool) -> dict | None:
        if self.remaining_seconds() <= 0:
            return _err("BudgetExceeded", "Session time budget exhausted")
        if count:
            rem = self.remaining_calls()
            if rem is not None and rem <= 0:
                return _err("BudgetExceeded", "Tool call budget exhausted; call submit_patch")
            self.tool_calls_used += 1
        return None

    def _finish(self, d: dict) -> str:
        total, rem = self.limits.tool_calls, self.remaining_calls()
        if total is not None and total >= 20 and rem is not None and rem <= 10:
            d["budget_warning"] = (f"Only {rem} tool call(s) remaining ({self.tool_calls_used}/{total} used). "
                                   "Finalize your edits and call submit_patch soon.")
        return json.dumps(d)

    def _path(self, filepath: str) -> Path:
        p = str(filepath or "").strip()
        for prefix in ("/workspace/", "/workspace"):
            if p.startswith(prefix):
                p = p[len(prefix):]
        p = p.lstrip("/")
        if ".." in Path(p).parts:
            raise ValueError("path traversal ('..') is not allowed")
        return self.sandbox.workspace / p

    # ---------------------------------------------------------------- tools
    def run_command(self, command: str) -> str:
        g = self._gate(True)
        if g:
            return json.dumps(g)
        timeout = min(self.limits.command_timeout, max(5, int(self.remaining_seconds())))
        r = self.sandbox.run(command, timeout=timeout)
        cap = self.limits.max_stdout_chars
        out, err = r.stdout[:cap], r.stderr[:cap]
        if r.timed_out:
            return self._finish(_err("TimeoutExceeded", f"Command exceeded {timeout}s timeout",
                                     {"stdout": out, "stderr": err}))
        if r.exit_code == 0:
            return self._finish({"status": "ok", "stdout": out + (("\n" + err) if err else ""), "exit_code": 0})
        return self._finish(_err("CommandError", f"Command failed with exit code {r.exit_code}",
                                 {"stdout": out, "stderr": err, "exit_code": r.exit_code}))

    def submit_patch(self) -> str:
        g = self._gate(False)
        if g:
            return json.dumps(g)
        patch = self.sandbox.diff()
        self.submitted_patch, self.patch_submitted = patch, True
        files = len({l[6:] for l in patch.splitlines() if l.startswith("+++ b/")})
        return json.dumps({"status": "ok", "patch_size": len(patch), "files_changed": files})

    def get_status(self) -> str:
        return json.dumps({
            "tool_calls_used": self.tool_calls_used, "patch_submitted": self.patch_submitted,
            "patch_size": len(self.submitted_patch), "tool_calls_remaining": self.remaining_calls(),
            "max_tool_calls": self.limits.tool_calls,
            "time_seconds_remaining": round(self.remaining_seconds(), 1),
            "max_time_minutes": self.limits.time_minutes,
            "agent_elapsed_seconds": round(time.time() - self.agent_start, 1),
            "max_turns": self.limits.turns, "command_timeout_seconds": self.limits.command_timeout,
        })

    def read_file(self, filepath: str, start_line: int | None = None, end_line: int | None = None) -> str:
        g = self._gate(True)
        if g:
            return json.dumps(g)
        try:
            p = self._path(filepath)
        except ValueError as e:
            return self._finish(_err("ValidationError", str(e)))
        if not p.is_file():
            return self._finish(_err("FileNotFoundError", f"File does not exist: {filepath}"))
        lines = p.read_text(errors="replace").split("\n")
        total = len(lines)
        s = max(1, int(start_line or 1))
        e = min(total, int(end_line) if end_line else total)
        e = min(e, s + self.limits.max_file_lines - 1)
        chunk, used, last = [], 0, s - 1
        for i in range(s - 1, e):
            add = len(lines[i]) + 1
            if used + add > self.limits.max_file_chars:
                break
            chunk.append(lines[i]); used += add; last = i + 1
        truncated = last < (min(total, int(end_line)) if end_line else total)
        return self._finish({"status": "ok", "filepath": str(filepath), "content": "\n".join(chunk),
                             "start_line": s, "end_line": last, "total_lines": total, "is_truncated": truncated})

    def edit_file(self, filepath: str, old_string: str, new_string: str, allow_multiple: bool = False) -> str:
        g = self._gate(True)
        if g:
            return json.dumps(g)
        try:
            p = self._path(filepath)
        except ValueError as e:
            return self._finish(_err("ValidationError", str(e)))
        if not p.is_file():
            return self._finish(_err("FileEditError", f"File does not exist: {filepath}"))
        before = p.read_text(errors="replace")
        if not before:
            return self._finish(_err("FileEditError", f"File is empty: {filepath}"))
        try:
            res = apply_replacement(before, old_string, new_string, bool(allow_multiple))
        except EditError as e:
            return self._finish(_err("FileEditError", f"{e} in {filepath}"))
        p.write_text(res.content)
        diff = "".join(difflib.unified_diff(before.replace("\r\n", "\n").splitlines(True), res.content.splitlines(True),
                                            f"a/{filepath}", f"b/{filepath}"))
        cap = self.limits.max_stdout_chars
        return self._finish({"status": "ok", "filepath": str(filepath), "occurrences": res.occurrences,
                             "strategy": res.strategy, "diff": diff[:cap], "is_truncated": len(diff) > cap})

    def write_file(self, filepath: str, content: str) -> str:
        g = self._gate(True)
        if g:
            return json.dumps(g)
        try:
            p = self._path(filepath)
        except ValueError as e:
            return self._finish(_err("ValidationError", str(e)))
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content if isinstance(content, str) else json.dumps(content))
        return self._finish({"status": "ok", "filepath": str(filepath), "size": p.stat().st_size})

    def get_code_neighbors(self, node: str, edge_type: str | None = None, max_neighbors: int = 50) -> str:
        g = self._gate(True)
        if g:
            return json.dumps(g)
        if not self.graph or not self.graph.nodes:
            return self._finish(_err("NoGraph", "No code graph available for this repository"))
        return self._finish(self.graph.neighbors(node, edge_type, max_neighbors))

    def search_similar_code(self, query: str, k: int = 10) -> str:
        g = self._gate(True)
        if g:
            return json.dumps(g)
        if not self.graph:
            return self._finish(_err("NoEmbeddings", "No embeddings available for this repository"))
        return self._finish(self.graph.similar(query, k))

    def get_code_subgraph(self, nodes: list[str]) -> str:
        g = self._gate(True)
        if g:
            return json.dumps(g)
        if not self.graph or not self.graph.nodes:
            return self._finish(_err("NoGraph", "No code graph available for this repository"))
        if isinstance(nodes, str):
            nodes = [nodes]
        return self._finish(self.graph.subgraph(nodes))

    # ------------------------------------------------------------- dispatch
    def registry(self) -> dict[str, Callable[..., str]]:
        names = ["run_command", "submit_patch", "get_status", "read_file", "edit_file", "write_file"]
        if self.graph is not None and self.graph.available:
            names += ["get_code_neighbors", "search_similar_code", "get_code_subgraph"]
        return {n: getattr(self, n) for n in names}

    def call(self, name: str, args: dict[str, Any]) -> str:
        """Execute like ADK would: unknown tool or bad arguments come back as an error result."""
        fn = self.registry().get(name)
        if fn is None:
            return json.dumps(_err("ToolNotFound", f"Function {name} is not found in the tools_dict."))
        try:
            return fn(**args)
        except TypeError as e:
            return json.dumps(_err("InvalidArguments", f"{name}: {e}"))
        except Exception as e:  # tool bug should not kill the session
            return json.dumps(_err(type(e).__name__, str(e)[:500]))


In [ ]:
%%writefile faultlines/src/faultlines/harness/verify.py
"""Phase 2 verification (HARNESS_README.md section 8).

Fresh workspace from the snapshot -> apply agent patch (fallback passes) ->
reset the files test_patch touches -> apply test_patch -> run pytest on those
files with the documented flags. resolved = pytest exit code 0.
"""

from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path

from .sandbox import SubprocessSandbox
from .tasks import DataLayout, Task, patch_targets

APPLY_PASSES = [
    "git apply --unsafe-paths {p}",
    "git apply -3 {p}",
    "git apply --unsafe-paths --ignore-space-change --ignore-whitespace {p}",
    "git apply --unsafe-paths --recount {p}",
    "git apply --unsafe-paths -p0 {p}",
    "patch -p1 --batch --forward --dry-run < {p} && patch -p1 --batch --forward < {p}",
    "patch -p0 --batch --forward --dry-run < {p} && patch -p0 --batch --forward < {p}",
]

PYTEST = ('PYTHONSAFEPATH=1 python3 -m pytest {targets} -p no:anyio -o timeout=0 '
          '-o norecursedirs=".* build dist venv" -o python_classes="Test* *Test" -q')


@dataclass
class VerifyResult:
    resolved: bool
    patch_applied: bool
    output: str
    seconds: float
    apply_pass: str = ""


def verify_task(task: Task, layout: DataLayout, patch: str, venv_path: Path, python: str = "python3",
                allow_online_fallback: bool = False, pytest_timeout: int = 1200) -> VerifyResult:
    t0 = time.time()
    sb = SubprocessSandbox(layout.snapshot(task.instance_id), layout.wheels if layout.wheels.exists() else None,
                           python=python, allow_online_fallback=allow_online_fallback, venv_path=venv_path)
    try:
        sb.setup(reuse_venv=venv_path)
        pf = sb.root / "agent.patch"
        pf.write_text(patch if patch.endswith("\n") else patch + "\n")
        used = ""
        for cmd in APPLY_PASSES:
            r = sb.run(cmd.format(p=pf), timeout=120, rewrite=False)
            if r.exit_code == 0:
                used = cmd.split(" {p}")[0].split(" <")[0]
                break
            sb._git("checkout", "--", ".", check=False)
        if not used:
            return VerifyResult(False, False, "Failed to apply agent_patch with all passes", time.time() - t0)
        targets = patch_targets(task.test_patch)
        if targets:
            sb._git("checkout", "HEAD", "--", *targets, check=False)
            sb._git("clean", "-f", "--", *targets, check=False)
        tp = sb.root / "test.patch"
        tp.write_text(task.test_patch if task.test_patch.endswith("\n") else task.test_patch + "\n")
        r = sb.run(f"git apply --unsafe-paths {tp} || patch -p1 --batch --forward < {tp}", timeout=120, rewrite=False)
        if r.exit_code != 0:
            return VerifyResult(False, True, "Failed to apply test_patch: " + r.stderr[-2000:], time.time() - t0, used)
        r = sb.run(PYTEST.format(targets=" ".join(targets) or "."), timeout=pytest_timeout, rewrite=False)
        out = (r.stdout + "\n" + r.stderr)[-20000:]
        if r.timed_out:
            out = "PYTEST TIMEOUT\n" + out
        return VerifyResult(r.exit_code == 0, True, out, time.time() - t0, used)
    finally:
        sb.cleanup()


In [ ]:
%%writefile faultlines/src/faultlines/stress.py
"""Stress set for RQ3: damage valid calls in known ways, check what the guard does.

Each case says what SHOULD happen:
    "repair" -> guard must return exactly the original call
    "refuse" -> guard must reject (repairing would mean inventing or dropping content)
"""

from __future__ import annotations

import json
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Callable

from .toolguard import ToolGuard, ToolRegistry, render_call, validate_call
from .toolguard.gemma_syntax import CALL_OPEN, CALL_CLOSE, STR, render_value
from .toolguard.repair import ARG_ALIASES, TOOL_ALIASES


@dataclass
class Case:
    kind: str
    text: str
    expect: str                   # "repair" | "refuse"
    name: str
    args: dict[str, Any]


def _drop_close_tag(name, args, reg, rng):
    return render_call(name, args)[: -len(CALL_CLOSE)], "repair"


def _truncate_in_string(name, args, reg, rng):
    keys = [k for k, v in args.items() if isinstance(v, str) and len(v) >= 8]
    if not keys:
        return None
    k = rng.choice(keys)
    full = render_call(name, args)
    marker = f"{k}:{STR}"
    start = full.index(marker) + len(marker)
    cut = start + rng.randint(1, len(args[k]) - 1)
    return full[:cut], "refuse"


def _json_args(name, args, reg, rng):
    return f"{CALL_OPEN}call:{name}{json.dumps(args)}{CALL_CLOSE}", "repair"


def _json_call(name, args, reg, rng):
    return f"{CALL_OPEN}{json.dumps({'name': name, 'arguments': args})}{CALL_CLOSE}", "repair"


def _tool_alias(name, args, reg, rng):
    aliases = [a for a, t in TOOL_ALIASES.items() if t == name]
    if not aliases:
        return None
    return render_call(rng.choice(aliases), args), "repair"


def _tool_typo(name, args, reg, rng):
    if len(name) < 6:
        return None
    i = rng.randint(1, len(name) - 2)
    typo = name[:i] + name[i + 1] + name[i] + name[i + 2:]
    if typo == name or typo in reg:
        return None
    return render_call(typo, args), "repair"


def _arg_alias(name, args, reg, rng):
    options = [(k, a) for k in args for a, t in ARG_ALIASES.items() if t == k and a not in args]
    if not options:
        return None
    k, a = rng.choice(options)
    bad = {(a if kk == k else kk): v for kk, v in args.items()}
    return render_call(name, bad), "repair"


def _int_as_string(name, args, reg, rng):
    keys = [k for k, v in args.items() if isinstance(v, int) and not isinstance(v, bool)]
    if not keys:
        return None
    k = rng.choice(keys)
    bad = dict(args)
    bad[k] = str(args[k])
    return render_call(name, bad), "repair"


def _extra_arg(name, args, reg, rng):
    bad = dict(args)
    bad["verbose"] = True
    return render_call(name, bad), "refuse"


def _missing_required(name, args, reg, rng):
    spec = reg.get(name)
    req = [k for k in (spec.required if spec else []) if k in args]
    if not req:
        return None
    drop = rng.choice(req)
    bad = {k: v for k, v in args.items() if k != drop}
    return render_call(name, bad), "refuse"


CORRUPTIONS: dict[str, Callable] = {
    "drop_close_tag": _drop_close_tag,
    "truncate_in_string": _truncate_in_string,
    "json_args": _json_args,
    "json_call": _json_call,
    "tool_alias": _tool_alias,
    "tool_typo": _tool_typo,
    "arg_alias": _arg_alias,
    "int_as_string": _int_as_string,
    "extra_arg": _extra_arg,
    "missing_required": _missing_required,
}


def sample_calls() -> list[tuple[str, dict[str, Any]]]:
    """Small built-in set for demos and tests. Use real calls from traces for the paper."""
    return [
        ("read_file", {"filepath": "src/utils/dates.py", "start_line": 40, "end_line": 120}),
        ("read_file", {"filepath": "tests/test_parser.py"}),
        ("run_command", {"command": "python -m pytest tests/test_parser.py -x -q"}),
        ("run_command", {"command": "grep -rn 'def parse_date' src/"}),
        ("edit_file", {"filepath": "src/utils/dates.py", "old_string": "if tz is None:\n    return dt",
                       "new_string": "if tz is None:\n    return dt.replace(tzinfo=UTC)"}),
        ("write_file", {"filepath": "/tmp/repro.py", "content": "from utils.dates import parse_date\nprint(parse_date('2020-01-01'))\n"}),
        ("search_similar_code", {"query": "parse_date", "k": 5}),
        ("get_code_neighbors", {"node": "utils.dates.parse_date"}),
    ]


def build_cases(calls: list[tuple[str, dict[str, Any]]], registry: ToolRegistry, seed: int = 0) -> list[Case]:
    rng = random.Random(seed)
    cases = []
    for name, args in calls:
        if validate_call(name, args, registry):
            continue  # only start from calls that are valid under this registry
        for kind, fn in CORRUPTIONS.items():
            res = fn(name, args, registry, rng)
            if res is None:
                continue
            text, expect = res
            cases.append(Case(kind, text, expect, name, args))
    return cases


def evaluate(cases: list[Case], registry: ToolRegistry, allow_lossy: bool = False) -> dict[str, Any]:
    per_kind: dict[str, Counter] = defaultdict(Counter)
    tot = Counter()
    for c in cases:
        g = ToolGuard(registry, allow_lossy=allow_lossy, detect_loops=False)
        r = g.check(c.text)
        repaired = r.status in ("valid", "repaired")
        exact = repaired and len(r.calls) == 1 and r.calls[0].name == c.name and r.calls[0].args == c.args
        correct = exact if c.expect == "repair" else not repaired
        per_kind[c.kind]["n"] += 1
        per_kind[c.kind]["correct"] += correct
        tot[f"expect_{c.expect}"] += 1
        tot[f"expect_{c.expect}_correct"] += correct
        tot["repaired_outputs"] += repaired
        tot["repaired_exact"] += exact
    return {
        "per_kind": {k: dict(v) for k, v in per_kind.items()},
        "coverage": tot["expect_repair_correct"] / max(1, tot["expect_repair"]),
        "precision": tot["repaired_exact"] / max(1, tot["repaired_outputs"]),
        "safe_refusal": tot["expect_refuse_correct"] / max(1, tot["expect_refuse"]),
        "n": len(cases),
    }


def report(result: dict[str, Any]) -> str:
    lines = ["| Corruption | Cases | Handled correctly |", "| --- | --- | --- |"]
    for k, v in sorted(result["per_kind"].items()):
        lines.append(f"| {k} | {v['n']} | {100 * v['correct'] / v['n']:.1f}% |")
    lines.append("")
    lines.append(f"Cases: {result['n']}  |  repair coverage {100 * result['coverage']:.1f}%  |  "
                 f"repair precision {100 * result['precision']:.1f}%  |  safe refusal {100 * result['safe_refusal']:.1f}%")
    return "\n".join(lines)


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/__init__.py
"""toolguard: parse, validate, and safely repair Gemma 4 tool calls."""

from .gemma_syntax import extract_calls, parse_body, render_call, GemmaSyntaxError, RawCall
from .guard import GuardResult, ToolGuard
from .loops import LoopDetector, find_loops
from .repair import RepairOutcome, ToolCall, repair_raw, feedback_message
from .schemas import ToolRegistry, ToolSpec, infer_registry
from .validate import CallError, Issue, validate_call

__all__ = [
    "extract_calls", "parse_body", "render_call", "GemmaSyntaxError", "RawCall",
    "GuardResult", "ToolGuard", "LoopDetector", "find_loops",
    "RepairOutcome", "ToolCall", "repair_raw", "feedback_message",
    "ToolRegistry", "ToolSpec", "infer_registry",
    "CallError", "Issue", "validate_call",
]


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/gemma_syntax.py
"""Parse and render Gemma 4's native tool-call syntax.

Gemma 4 emits tool calls as special-token-delimited text, e.g.

    <|tool_call>call:read_file{path:<|"|>src/app.py<|"|>,start_line:10}<tool_call|>

Keys are bare identifiers, strings are wrapped in the <|"|> token, numbers and
booleans are bare, and objects / arrays use {} and [].
Source: https://ai.google.dev/gemma/docs/capabilities/function-calling
(verify against the chat template of the exact checkpoint you run).

This module never guesses content. It reports *what* is wrong so that
`repair.py` can decide what is safe to fix.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

CALL_OPEN = "<|tool_call>"
CALL_CLOSE = "<tool_call|>"
STR = '<|"|>'

_IDENT = re.compile(r"[A-Za-z_][A-Za-z0-9_.\-]*")


class GemmaSyntaxError(ValueError):
    """Raised when a call body cannot be parsed. `code` is a stable reason id."""

    def __init__(self, code: str, detail: str, pos: int = -1):
        super().__init__(f"{code}: {detail} (at {pos})")
        self.code = code
        self.detail = detail
        self.pos = pos


@dataclass
class RawCall:
    """One `<|tool_call>...` region found in model output."""

    body: str               # text between the open tag and the close tag (or end of text)
    start: int              # offset of the open tag in the original text
    closed: bool            # False when the close tag is missing (usually MAX_TOKENS truncation)


@dataclass
class ParsedBody:
    name: str
    args: dict[str, Any]
    warnings: list[str] = field(default_factory=list)  # lenient-parse notes, e.g. BARE_STRING


def extract_calls(text: str) -> list[RawCall]:
    """Find every tool-call region in `text`, including an unclosed trailing one."""
    calls: list[RawCall] = []
    i = 0
    while True:
        start = text.find(CALL_OPEN, i)
        if start < 0:
            break
        body_start = start + len(CALL_OPEN)
        # A string literal may legally contain the close tag's characters only if the
        # model emitted it inside <|"|>...<|"|>; scan string-aware.
        end = _find_close(text, body_start)
        if end < 0:
            calls.append(RawCall(body=text[body_start:], start=start, closed=False))
            break
        calls.append(RawCall(body=text[body_start:end], start=start, closed=True))
        i = end + len(CALL_CLOSE)
    return calls


def _find_close(text: str, pos: int) -> int:
    """String-aware search for the close tag.

    Falls back to a plain search when string delimiters are unbalanced, so a call
    that is closed but has a broken string is reported as MALFORMED, not TRUNCATED.
    """
    end = _find_close_string_aware(text, pos)
    if end >= 0:
        return end
    plain = text.find(CALL_CLOSE, pos)
    next_open = text.find(CALL_OPEN, pos)
    if plain >= 0 and (next_open < 0 or plain < next_open):
        return plain
    return -1


def _find_close_string_aware(text: str, pos: int) -> int:
    in_str = False
    i = pos
    while i < len(text):
        if text.startswith(STR, i):
            in_str = not in_str
            i += len(STR)
            continue
        if not in_str and text.startswith(CALL_CLOSE, i):
            return i
        if not in_str and text.startswith(CALL_OPEN, i):
            # A new call opened before this one closed: treat this one as unclosed.
            return -1
        i += 1
    return -1


def parse_body(body: str, allow_trailing: bool = False) -> ParsedBody:
    """Parse `call:NAME{...}` into a name and an argument dict (strict Gemma syntax).

    With `allow_trailing`, text after the closing brace is ignored and noted as a
    TRAILING_TEXT warning (useful when the close tag is missing and the model kept
    talking after a complete call).
    """
    s = body.strip()
    if not s.startswith("call:"):
        raise GemmaSyntaxError("MISSING_CALL_PREFIX", "body does not start with 'call:'", 0)
    s = s[len("call:"):]
    m = _IDENT.match(s)
    if not m:
        raise GemmaSyntaxError("BAD_TOOL_NAME", "no tool name after 'call:'", len("call:"))
    name = m.group(0)
    rest = s[m.end():].strip()
    if rest == "":
        return ParsedBody(name=name, args={})  # zero-arg call without braces
    p = _Parser(rest)
    args = p.parse_object()
    p.skip_ws()
    if p.i != len(p.t):
        if not allow_trailing:
            raise GemmaSyntaxError("TRAILING_TEXT", repr(p.t[p.i:p.i + 40]), p.i)
        p.warnings.append("TRAILING_TEXT")
    if not isinstance(args, dict):
        raise GemmaSyntaxError("ARGS_NOT_OBJECT", "arguments must be an object", 0)
    return ParsedBody(name=name, args=args, warnings=p.warnings)


class _Parser:
    def __init__(self, text: str):
        self.t = text
        self.i = 0
        self.warnings: list[str] = []

    def skip_ws(self) -> None:
        while self.i < len(self.t) and self.t[self.i] in " \t\r\n":
            self.i += 1

    def peek(self) -> str:
        return self.t[self.i] if self.i < len(self.t) else ""

    def expect(self, ch: str) -> None:
        self.skip_ws()
        if self.peek() != ch:
            code = "UNEXPECTED_END" if self.i >= len(self.t) else "UNEXPECTED_CHAR"
            raise GemmaSyntaxError(code, f"expected {ch!r}, got {self.peek()!r}", self.i)
        self.i += 1

    def parse_value(self) -> Any:
        self.skip_ws()
        if self.t.startswith(STR, self.i):
            return self.parse_string()
        c = self.peek()
        if c == "{":
            return self.parse_object()
        if c == "[":
            return self.parse_array()
        if c == "":
            raise GemmaSyntaxError("UNEXPECTED_END", "value expected", self.i)
        return self.parse_bare()

    def parse_string(self) -> str:
        self.i += len(STR)
        j = self.t.find(STR, self.i)
        if j < 0:
            raise GemmaSyntaxError("UNTERMINATED_STRING", "missing closing <|\"|>", self.i)
        val = self.t[self.i:j]
        self.i = j + len(STR)
        return val

    def parse_bare(self) -> Any:
        j = self.i
        while j < len(self.t) and self.t[j] not in ",}]":
            if self.t.startswith(STR, j):
                break
            j += 1
        tok = self.t[self.i:j].strip()
        self.i = j
        if tok == "":
            raise GemmaSyntaxError("EMPTY_VALUE", "empty value", self.i)
        low = tok.lower()
        if low == "true":
            return True
        if low == "false":
            return False
        if low in ("null", "none"):
            return None
        try:
            return int(tok)
        except ValueError:
            pass
        try:
            return float(tok)
        except ValueError:
            pass
        # Lenient: an unquoted string. Record it; repair.py treats it as a warning.
        self.warnings.append(f"BARE_STRING:{tok[:40]}")
        return tok

    def parse_key(self) -> str:
        self.skip_ws()
        if self.t.startswith(STR, self.i):
            return self.parse_string()
        c = self.peek()
        if c in "\"'":
            j = self.t.find(c, self.i + 1)
            if j < 0:
                raise GemmaSyntaxError("UNTERMINATED_KEY", "key quote not closed", self.i)
            key = self.t[self.i + 1:j]
            self.i = j + 1
            self.warnings.append("QUOTED_KEY")
            return key
        m = _IDENT.match(self.t, self.i)
        if not m:
            code = "UNEXPECTED_END" if self.i >= len(self.t) else "BAD_KEY"
            raise GemmaSyntaxError(code, f"bad key near {self.t[self.i:self.i + 20]!r}", self.i)
        self.i = m.end()
        return m.group(0)

    def parse_object(self) -> dict[str, Any]:
        self.expect("{")
        out: dict[str, Any] = {}
        self.skip_ws()
        if self.peek() == "}":
            self.i += 1
            return out
        while True:
            key = self.parse_key()
            self.expect(":")
            if key in out:
                self.warnings.append(f"DUPLICATE_KEY:{key}")
            out[key] = self.parse_value()
            self.skip_ws()
            c = self.peek()
            if c == ",":
                self.i += 1
                self.skip_ws()
                if self.peek() == "}":  # trailing comma
                    self.warnings.append("TRAILING_COMMA")
                    self.i += 1
                    return out
                continue
            if c == "}":
                self.i += 1
                return out
            code = "UNEXPECTED_END" if c == "" else "UNEXPECTED_CHAR"
            raise GemmaSyntaxError(code, f"expected ',' or '}}', got {c!r}", self.i)

    def parse_array(self) -> list[Any]:
        self.expect("[")
        out: list[Any] = []
        self.skip_ws()
        if self.peek() == "]":
            self.i += 1
            return out
        while True:
            out.append(self.parse_value())
            self.skip_ws()
            c = self.peek()
            if c == ",":
                self.i += 1
                continue
            if c == "]":
                self.i += 1
                return out
            code = "UNEXPECTED_END" if c == "" else "UNEXPECTED_CHAR"
            raise GemmaSyntaxError(code, f"expected ',' or ']', got {c!r}", self.i)


def render_value(v: Any) -> str:
    if isinstance(v, bool):
        return "true" if v else "false"
    if v is None:
        return "null"
    if isinstance(v, (int, float)):
        return repr(v)
    if isinstance(v, str):
        return f"{STR}{v}{STR}"
    if isinstance(v, list):
        return "[" + ",".join(render_value(x) for x in v) + "]"
    if isinstance(v, dict):
        return "{" + ",".join(f"{k}:{render_value(x)}" for k, x in v.items()) + "}"
    raise TypeError(f"cannot render {type(v)}")


def render_call(name: str, args: dict[str, Any]) -> str:
    """Render a call in canonical Gemma 4 syntax (inverse of extract_calls + parse_body)."""
    return f"{CALL_OPEN}call:{name}{render_value(args)}{CALL_CLOSE}"


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/guard.py
"""One entry point: check (and optionally repair) a model turn.

    guard = ToolGuard(ToolRegistry.default_harness())
    result = guard.check(model_output_text)
    if result.status in ("valid", "repaired"):
        execute(result.calls)
    else:
        send_back(result.feedback)
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any

from .gemma_syntax import extract_calls
from .loops import LoopDetector, LoopSignal
from .repair import (RepairOutcome, ToolCall, feedback_message, parse_structured_args,
                     repair_parsed, repair_raw)
from .schemas import ToolRegistry
from .validate import CallError, Issue


@dataclass
class GuardResult:
    status: str                                   # valid | repaired | rejected | no_call
    calls: list[ToolCall] = field(default_factory=list)
    outcomes: list[RepairOutcome] = field(default_factory=list)
    loop: LoopSignal | None = None
    feedback: str = ""

    @property
    def issues(self) -> list[Issue]:
        """Problems present in the model's ORIGINAL output (before repair)."""
        out = [i for o in self.outcomes for i in o.initial]
        if not self.outcomes and self.status == "no_call":
            out.append(Issue(CallError.NO_CALL, "no tool call in turn"))
        if self.loop is not None:
            out.append(Issue(CallError.REPEATED_CALL, f"{self.loop.kind} x{self.loop.count}"))
        return out

    def to_dict(self) -> dict[str, Any]:
        return {
            "status": self.status,
            "calls": [c.to_dict() for c in self.calls],
            "issues": [i.to_dict() for i in self.issues],
            "repairs": [a for o in self.outcomes for a in o.applied],
            "lossy": any(o.lossy for o in self.outcomes),
            "loop": None if self.loop is None else self.loop.__dict__,
            "feedback": self.feedback,
        }


class ToolGuard:
    def __init__(self, registry: ToolRegistry, allow_lossy: bool = False, detect_loops: bool = True, **loop_kw):
        self.registry = registry
        self.allow_lossy = allow_lossy
        self.loops = LoopDetector(**loop_kw) if detect_loops else None

    def check(self, text: str) -> GuardResult:
        raws = extract_calls(text or "")
        return self._finish([repair_raw(r, self.registry, self.allow_lossy) for r in raws])

    def check_structured(self, name: str, arguments: Any) -> GuardResult:
        """For servers that already parsed the call (OpenAI-style `tool_calls`)."""
        args, applied = parse_structured_args(arguments)
        if args is None:
            bad = Issue(CallError.MALFORMED_SYNTAX, "arguments are not a JSON object")
            return self._finish([RepairOutcome(call=None, initial=[bad], remaining=[bad])])
        syntax = [Issue(CallError.MALFORMED_SYNTAX, "invalid JSON arguments")] if applied else []
        return self._finish([repair_parsed(ToolCall(name, args), self.registry, self.allow_lossy,
                                           applied=applied, syntax_issues=syntax)])

    def _finish(self, outcomes: list[RepairOutcome]) -> GuardResult:
        if not outcomes:
            return GuardResult(status="no_call",
                               feedback="No tool call found. Call exactly one tool, or submit_patch if done.")
        if any(not o.ok for o in outcomes):
            bad = next(o for o in outcomes if not o.ok)
            return GuardResult(status="rejected", outcomes=outcomes, feedback=feedback_message(bad, self.registry))
        calls = [o.call for o in outcomes if o.call is not None]
        loop = None
        if self.loops is not None:
            for c in calls:
                loop = self.loops.observe(c.key()) or loop
        status = "repaired" if any(o.applied for o in outcomes) else "valid"
        fb = ""
        if loop is not None:
            fb = ("You have repeated the same action several times without progress. "
                  "Try a different approach: re-read the error, check the file content, or change strategy.")
        return GuardResult(status=status, calls=calls, outcomes=outcomes, loop=loop, feedback=fb)


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/harness_tools.json
{
  "_comment": "The 9 swegemma tools with the exact signatures published in the competition's HARNESS_README.md and Overview (Gemma 4 Developer Agent Competition, Sep 2026).",
  "tools": [
    {"name": "run_command", "description": "Execute a shell command in /bin/bash -c inside /workspace. 300 s timeout; stdout/stderr truncated to 5000 characters.",
     "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}},
    {"name": "submit_patch", "description": "Stage untracked file intents (git add -N .) and capture git diff HEAD from /workspace. Call last; ends the session.",
     "parameters": {"type": "object", "properties": {}, "required": []}},
    {"name": "get_status", "description": "Return live budget consumption and patch status.",
     "parameters": {"type": "object", "properties": {}, "required": []}},
    {"name": "read_file", "description": "Read a file from /workspace with 1-indexed inclusive line slicing (max 150 lines / 10000 characters per call).",
     "parameters": {"type": "object", "properties": {"filepath": {"type": "string"}, "start_line": {"type": "integer"}, "end_line": {"type": "integer"}}, "required": ["filepath"]}},
    {"name": "edit_file", "description": "Replace old_string with new_string in an existing non-empty file inside /workspace.",
     "parameters": {"type": "object", "properties": {"filepath": {"type": "string"}, "old_string": {"type": "string"}, "new_string": {"type": "string"}, "allow_multiple": {"type": "boolean"}}, "required": ["filepath", "old_string", "new_string"]}},
    {"name": "write_file", "description": "Create or overwrite a file at /workspace/<filepath>, creating parent directories.",
     "parameters": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "get_code_neighbors", "description": "Find incoming and outgoing neighbors of a symbol in the repository call/dependency graph.",
     "parameters": {"type": "object", "properties": {"node": {"type": "string"}, "edge_type": {"type": "string"}, "max_neighbors": {"type": "integer"}}, "required": ["node"]}},
    {"name": "search_similar_code", "description": "Find the top-k graph nodes most similar to a symbol name using the pre-computed embeddings.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}, "k": {"type": "integer"}}, "required": ["query"]}},
    {"name": "get_code_subgraph", "description": "Extract the induced subgraph (nodes and interconnecting edges) for a list of symbols.",
     "parameters": {"type": "object", "properties": {"nodes": {"type": "array"}}, "required": ["nodes"]}}
  ]
}


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/loops.py
"""Detect an agent stuck repeating itself.

Two patterns, both common in small-model traces:
  * the same call N times within a short window (e.g. re-reading one file), and
  * a short cycle repeated (A B A B A B, e.g. edit fails -> read -> edit fails ...).
"""

from __future__ import annotations

from collections import deque
from dataclasses import dataclass


@dataclass
class LoopSignal:
    kind: str       # "repeat" or "cycle"
    period: int     # 1 for repeat, cycle length otherwise
    count: int      # how many times it repeated


class LoopDetector:
    def __init__(self, window: int = 12, repeat_threshold: int = 3, cycle_repeats: int = 3, max_period: int = 3):
        self.history: deque[str] = deque(maxlen=window)
        self.repeat_threshold = repeat_threshold
        self.cycle_repeats = cycle_repeats
        self.max_period = max_period

    def observe(self, call_key: str) -> LoopSignal | None:
        self.history.append(call_key)
        h = list(self.history)
        # consecutive identical calls
        run = 1
        for prev in reversed(h[:-1]):
            if prev != call_key:
                break
            run += 1
        if run >= self.repeat_threshold:
            return LoopSignal("repeat", 1, run)
        # cycles of period 2..max_period ending at the latest call
        for period in range(2, self.max_period + 1):
            need = period * self.cycle_repeats
            if len(h) < need:
                continue
            tail = h[-need:]
            unit = tail[:period]
            if len(set(unit)) == period and all(tail[i] == unit[i % period] for i in range(need)):
                return LoopSignal("cycle", period, self.cycle_repeats)
        return None

    def reset(self) -> None:
        self.history.clear()


def find_loops(call_keys: list[str | None], **kw) -> list[tuple[int, LoopSignal]]:
    """Run a detector over a whole trace. `None` entries (no valid call) are skipped."""
    det = LoopDetector(**kw)
    hits = []
    for i, k in enumerate(call_keys):
        if k is None:
            continue
        sig = det.observe(k)
        if sig is not None:
            hits.append((i, sig))
    return hits


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/repair.py
"""Safe, deterministic repair of broken tool calls.

Design rule: repair only what the model clearly meant; never invent content.

* A call cut off inside a string (e.g. half of a write_file body) is REFUSED,
  because completing it would write a partial file.
* Renames (tool or argument) happen only through an alias table or a very close
  spelling match, and only to a name that exists and is not already used.
* Dropping an argument loses information, so it is off unless allow_lossy=True.

Every repair is recorded by name so the paper can report exactly what was done.
"""

from __future__ import annotations

import ast
import difflib
import json
import re
from dataclasses import dataclass, field
from typing import Any

from .gemma_syntax import GemmaSyntaxError, RawCall, parse_body, render_value, CALL_OPEN, CALL_CLOSE
from .schemas import ToolRegistry
from .validate import CallError, Issue, validate_call, type_ok

TOOL_ALIASES = {
    "bash": "run_command", "shell": "run_command", "execute": "run_command", "execute_bash": "run_command",
    "run": "run_command", "cmd": "run_command", "terminal": "run_command", "run_bash": "run_command",
    "cat": "read_file", "view": "read_file", "open_file": "read_file", "view_file": "read_file",
    "read": "read_file", "open": "read_file",
    "str_replace": "edit_file", "str_replace_editor": "edit_file", "replace": "edit_file",
    "edit": "edit_file", "apply_edit": "edit_file", "replace_in_file": "edit_file",
    "create_file": "write_file", "create": "write_file", "write": "write_file", "save_file": "write_file",
    "submit": "submit_patch", "finish": "submit_patch", "done": "submit_patch", "submit_solution": "submit_patch",
    "status": "get_status",
}

ARG_ALIASES = {
    # read_file / edit_file / write_file take `filepath` in the swegemma harness
    "path": "filepath", "file": "filepath", "filename": "filepath", "file_path": "filepath",
    "path_to_file": "filepath",
    "cmd": "command", "bash": "command", "script": "command",
    "old": "old_string", "old_str": "old_string", "old_text": "old_string", "search": "old_string",
    "find": "old_string", "original": "old_string",
    "new": "new_string", "new_str": "new_string", "new_text": "new_string", "replace": "new_string",
    "replacement": "new_string",
    "replace_all": "allow_multiple", "multiple": "allow_multiple",
    "text": "content", "contents": "content", "file_text": "content", "data": "content", "body": "content",
    "start": "start_line", "line_start": "start_line", "from_line": "start_line",
    "end": "end_line", "line_end": "end_line", "to_line": "end_line",
    "top_k": "k", "limit": "k", "n": "k", "max_results": "k", "q": "query",
    "symbol": "node", "name": "node", "limit_neighbors": "max_neighbors",
}

_TRUNCATION_CODES = {"UNTERMINATED_STRING", "UNEXPECTED_END", "UNTERMINATED_KEY"}
_INT_RE = re.compile(r"^-?\d+$")


@dataclass
class ToolCall:
    name: str
    args: dict[str, Any]

    def key(self) -> str:
        """Canonical string used for loop detection."""
        return self.name + json.dumps(self.args, sort_keys=True, default=str)

    def to_dict(self) -> dict[str, Any]:
        return {"name": self.name, "args": self.args}


@dataclass
class RepairOutcome:
    call: ToolCall | None                    # best-effort call (None if nothing usable)
    initial: list[Issue] = field(default_factory=list)    # problems found before repair
    remaining: list[Issue] = field(default_factory=list)  # problems left after repair
    applied: list[str] = field(default_factory=list)      # names of repairs applied
    lossy: bool = False

    @property
    def ok(self) -> bool:
        return self.call is not None and not self.remaining

    @property
    def status(self) -> str:
        if not self.ok:
            return "rejected"
        return "repaired" if self.applied else "valid"


# ---------------------------------------------------------------- parsing layer

def _parse_with_fallbacks(raw: RawCall) -> tuple[ToolCall | None, list[str], Issue | None]:
    """Return (call, repairs_applied, fatal_issue)."""
    applied: list[str] = []
    body = raw.body

    if not raw.closed:
        try:
            pb = parse_body(body, allow_trailing=True)
        except GemmaSyntaxError as e:
            if e.code in _TRUNCATION_CODES:
                return None, [], Issue(CallError.TRUNCATED_CALL, f"cut off: {e.code}")
            return None, [], Issue(CallError.MALFORMED_SYNTAX, f"{e.code}: {e.detail}")
        if _looks_like_json(pb.warnings):
            json_call = _json_args(body.strip())
            if json_call is not None:
                return json_call, ["CLOSE_TAG", "JSON_ARGS"], None
        applied.append("CLOSE_TAG")
        applied += [f"LENIENT:{w.split(':')[0]}" for w in pb.warnings]
        return ToolCall(pb.name, pb.args), applied, None

    try:
        pb = parse_body(body)
        if _looks_like_json(pb.warnings):
            # e.g. call:read_file{"path": "a.py"} parses leniently but with quotes kept
            json_call = _json_args(body.strip())
            if json_call is not None:
                return json_call, ["JSON_ARGS"], None
        applied += [f"LENIENT:{w.split(':')[0]}" for w in pb.warnings]
        return ToolCall(pb.name, pb.args), applied, None
    except GemmaSyntaxError as e:
        first_error = e

    s = body.strip()
    # (a) whole body is a JSON / python dict: {"name": ..., "arguments": {...}}
    obj = _loads_any(s)
    if isinstance(obj, dict) and isinstance(obj.get("name"), str):
        args = obj.get("arguments", obj.get("args", obj.get("parameters", {})))
        if isinstance(args, str):
            args = _loads_any(args)
        if isinstance(args, dict):
            return ToolCall(obj["name"], args), ["JSON_CALL"], None

    # (c) call:NAME followed by JSON or python-literal arguments
    json_call = _json_args(s)
    if json_call is not None:
        return json_call, ["JSON_ARGS"], None

    # (b) missing 'call:' prefix but otherwise Gemma syntax: NAME{...}
    if not s.startswith("call:") and re.match(r"^[A-Za-z_][\w.\-]*\s*\{", s):
        try:
            pb = parse_body("call:" + s)
            return ToolCall(pb.name, pb.args), ["ADD_CALL_PREFIX"], None
        except GemmaSyntaxError:
            pass

    return None, [], Issue(CallError.MALFORMED_SYNTAX, f"{first_error.code}: {first_error.detail}")


def _looks_like_json(warnings: list[str]) -> bool:
    return any(w.startswith(("QUOTED_KEY", "BARE_STRING")) for w in warnings)


def _json_args(s: str) -> ToolCall | None:
    m = re.match(r"^(?:call:)?\s*([A-Za-z_][\w.\-]*)\s*(\{.*\})\s*$", s, re.S)
    if not m:
        return None
    args = _loads_any(m.group(2))
    return ToolCall(m.group(1), args) if isinstance(args, dict) else None


def _loads_any(s: str) -> Any:
    try:
        return json.loads(s)
    except (json.JSONDecodeError, TypeError):
        pass
    try:
        # json with trailing commas
        return json.loads(re.sub(r",\s*([}\]])", r"\1", s))
    except (json.JSONDecodeError, TypeError):
        pass
    try:
        return ast.literal_eval(s)  # safe: literals only
    except (ValueError, SyntaxError, TypeError, MemoryError, RecursionError):
        return None


# ---------------------------------------------------------------- schema layer

def _coerce(value: Any, expected: str | list[str]) -> tuple[bool, Any]:
    kinds = expected if isinstance(expected, list) else [expected]
    for t in kinds:
        if t == "integer" and isinstance(value, str) and _INT_RE.match(value.strip()):
            return True, int(value.strip())
        if t == "integer" and isinstance(value, float) and value.is_integer():
            return True, int(value)
        if t == "number" and isinstance(value, str):
            try:
                return True, float(value.strip())
            except ValueError:
                pass
        if t == "boolean" and isinstance(value, str) and value.strip().lower() in ("true", "false"):
            return True, value.strip().lower() == "true"
        if t == "string" and isinstance(value, (int, float)) and not isinstance(value, bool):
            return True, str(value)
        if t in ("array", "object") and isinstance(value, str):
            parsed = _loads_any(value)
            if type_ok(parsed, t):
                return True, parsed
        if t == "array" and not isinstance(value, (list, dict)) and value is not None:
            return True, [value]
    return False, value


def repair_call(call: ToolCall, registry: ToolRegistry, allow_lossy: bool = False) -> tuple[ToolCall, list[str], bool]:
    applied: list[str] = []
    lossy = False
    name = call.name
    args = dict(call.args)

    if name not in registry:
        target = TOOL_ALIASES.get(name.lower())
        if target in registry:
            applied.append(f"TOOL_ALIAS:{name}->{target}")
            name = target
        else:
            close = difflib.get_close_matches(name, registry.names, n=1, cutoff=0.8)
            if close:
                applied.append(f"TOOL_TYPO:{name}->{close[0]}")
                name = close[0]
    spec = registry.get(name)
    if spec is None:
        return ToolCall(name, args), applied, lossy

    for k in list(args):
        if k in spec.params:
            continue
        free = [p for p in spec.params if p not in args]
        target = ARG_ALIASES.get(k.lower())
        if target not in free:
            close = difflib.get_close_matches(k, free, n=1, cutoff=0.8)
            target = close[0] if close else None
            kind = "ARG_TYPO"
        else:
            kind = "ARG_ALIAS"
        if target:
            args[target] = args.pop(k)
            applied.append(f"{kind}:{k}->{target}")

    for k, v in list(args.items()):
        p = spec.params.get(k)
        if p is not None and not type_ok(v, p.type):
            ok, nv = _coerce(v, p.type)
            if ok:
                args[k] = nv
                applied.append(f"COERCE:{k}")

    if allow_lossy:
        for k in [k for k in args if k not in spec.params]:
            args.pop(k)
            applied.append(f"DROP_ARG:{k}")
            lossy = True

    return ToolCall(name, args), applied, lossy


def repair_raw(raw: RawCall, registry: ToolRegistry, allow_lossy: bool = False) -> RepairOutcome:
    call, applied, fatal = _parse_with_fallbacks(raw)
    if fatal is not None:
        return RepairOutcome(call=None, initial=[fatal], remaining=[fatal])
    assert call is not None
    syntax: list[Issue] = []
    if not raw.closed:
        syntax.append(Issue(CallError.TRUNCATED_CALL, "close tag missing; arguments complete"))
    elif any(a in applied for a in ("JSON_CALL", "ADD_CALL_PREFIX", "JSON_ARGS")):
        syntax.append(Issue(CallError.MALFORMED_SYNTAX, "non-Gemma argument syntax"))
    return repair_parsed(call, registry, allow_lossy, applied=applied, syntax_issues=syntax)


def repair_parsed(call: ToolCall, registry: ToolRegistry, allow_lossy: bool = False,
                  applied: list[str] | None = None, syntax_issues: list[Issue] | None = None) -> RepairOutcome:
    """Schema-level validation and repair of an already-parsed call."""
    initial = list(syntax_issues or []) + validate_call(call.name, call.args, registry)
    fixed, more, lossy = repair_call(call, registry, allow_lossy)
    remaining = validate_call(fixed.name, fixed.args, registry)
    return RepairOutcome(call=fixed, initial=initial, remaining=remaining,
                         applied=list(applied or []) + more, lossy=lossy)


def parse_structured_args(arguments: Any) -> tuple[dict[str, Any] | None, list[str]]:
    """Arguments from an OpenAI-style tool_call: a dict, or a JSON string (maybe broken)."""
    if isinstance(arguments, dict):
        return arguments, []
    if arguments is None or (isinstance(arguments, str) and not arguments.strip()):
        return {}, []
    try:
        val = json.loads(arguments)
        return (val, []) if isinstance(val, dict) else (None, [])
    except (json.JSONDecodeError, TypeError):
        val = _loads_any(arguments)
        return (val, ["LENIENT_JSON"]) if isinstance(val, dict) else (None, [])


# ---------------------------------------------------------------- feedback

def example_call(tool: str, registry: ToolRegistry) -> str:
    spec = registry.get(tool)
    if spec is None:
        return ""
    placeholder = {"string": "...", "integer": 1, "number": 1.0, "boolean": True, "array": [], "object": {}}
    args = {}
    for p in spec.params.values():
        if p.required:
            t = p.type[0] if isinstance(p.type, list) else p.type
            args[p.name] = placeholder.get(t, "...")
    return f"{CALL_OPEN}call:{tool}{render_value(args)}{CALL_CLOSE}"


def feedback_message(outcome: RepairOutcome, registry: ToolRegistry) -> str:
    """Short, specific error text to send back to the model (for your own scaffold)."""
    if outcome.ok:
        return ""
    lines = ["Your last tool call was not executed."]
    for iss in outcome.remaining:
        lines.append(f"- {iss.code.value}: {iss.detail}")
    codes = {i.code for i in outcome.remaining}
    if CallError.TRUNCATED_CALL in codes:
        lines.append("Your output was cut off. Make the call shorter: edit a smaller region, "
                     "or write the file in parts.")
    if CallError.UNKNOWN_TOOL in codes:
        lines.append(f"Available tools: {', '.join(registry.names)}.")
    if outcome.call is not None and outcome.call.name in registry:
        lines.append("Correct form: " + example_call(outcome.call.name, registry))
    return "\n".join(lines)


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/schemas.py
"""Tool specifications: load, look up, and infer from observed calls.

The spec format is the common function-declaration shape used by Gemma, OpenAI
and ADK:

    {"name": "read_file",
     "parameters": {"type": "object",
                    "properties": {"path": {"type": "string"}},
                    "required": ["path"]}}
"""

from __future__ import annotations

import json
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable

JSON_TYPES = ("string", "integer", "number", "boolean", "array", "object", "null")


@dataclass
class ParamSpec:
    name: str
    type: str | list[str] = "string"
    required: bool = False
    enum: list[Any] | None = None


@dataclass
class ToolSpec:
    name: str
    params: dict[str, ParamSpec] = field(default_factory=dict)
    description: str = ""

    @property
    def required(self) -> list[str]:
        return [p.name for p in self.params.values() if p.required]

    @classmethod
    def from_declaration(cls, decl: dict[str, Any]) -> "ToolSpec":
        params_obj = decl.get("parameters") or decl.get("input_schema") or {}
        props = params_obj.get("properties") or {}
        req = set(params_obj.get("required") or [])
        params = {
            k: ParamSpec(name=k, type=v.get("type", "string"), required=k in req, enum=v.get("enum"))
            for k, v in props.items()
        }
        return cls(name=decl["name"], params=params, description=decl.get("description", ""))

    def to_declaration(self) -> dict[str, Any]:
        props: dict[str, Any] = {}
        for p in self.params.values():
            d: dict[str, Any] = {"type": p.type}
            if p.enum is not None:
                d["enum"] = p.enum
            props[p.name] = d
        return {
            "name": self.name,
            "description": self.description,
            "parameters": {"type": "object", "properties": props, "required": self.required},
        }


class ToolRegistry:
    def __init__(self, specs: Iterable[ToolSpec]):
        self.specs: dict[str, ToolSpec] = {s.name: s for s in specs}

    def __contains__(self, name: str) -> bool:
        return name in self.specs

    def get(self, name: str) -> ToolSpec | None:
        return self.specs.get(name)

    @property
    def names(self) -> list[str]:
        return list(self.specs)

    @classmethod
    def from_file(cls, path: str | Path) -> "ToolRegistry":
        data = json.loads(Path(path).read_text())
        decls = data["tools"] if isinstance(data, dict) else data
        return cls(ToolSpec.from_declaration(d) for d in decls)

    @classmethod
    def default_harness(cls) -> "ToolRegistry":
        """The 9 harness tools with GUESSED argument names. Replace after week 1."""
        return cls.from_file(Path(__file__).with_name("harness_tools.json"))

    def dump(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps({"tools": [s.to_declaration() for s in self.specs.values()]}, indent=2))


def _json_type(v: Any) -> str:
    if isinstance(v, bool):
        return "boolean"
    if isinstance(v, int):
        return "integer"
    if isinstance(v, float):
        return "number"
    if isinstance(v, str):
        return "string"
    if isinstance(v, list):
        return "array"
    if isinstance(v, dict):
        return "object"
    return "null"


def infer_registry(calls: Iterable[tuple[str, dict[str, Any]]], min_support: float = 0.95) -> ToolRegistry:
    """Infer tool specs from calls the harness ACCEPTED (i.e. that got a normal result).

    An argument is marked required when it appears in >= `min_support` of that
    tool's calls. Use this when the official declarations are not available.
    """
    per_tool: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for name, args in calls:
        per_tool[name].append(args)
    specs = []
    for name, arg_list in per_tool.items():
        n = len(arg_list)
        seen: Counter[str] = Counter()
        types: dict[str, Counter[str]] = defaultdict(Counter)
        for args in arg_list:
            for k, v in args.items():
                seen[k] += 1
                types[k][_json_type(v)] += 1
        params = {
            k: ParamSpec(name=k, type=types[k].most_common(1)[0][0], required=seen[k] / n >= min_support)
            for k in seen
        }
        specs.append(ToolSpec(name=name, params=params, description=f"inferred from {n} calls"))
    return ToolRegistry(specs)


In [ ]:
%%writefile faultlines/src/faultlines/toolguard/validate.py
"""Classify what is wrong with a tool call. One stable code per problem.

Call-level error codes (these become the step labels in FaultLines-Traces):

    NO_CALL            the turn contains no tool call at all
    TRUNCATED_CALL     <|tool_call> opened but never closed (usually MAX_TOKENS)
    MALFORMED_SYNTAX   the call body does not parse as Gemma syntax
    UNKNOWN_TOOL       tool name not in the registry
    MISSING_ARG        a required argument is absent
    UNEXPECTED_ARG     an argument the tool does not declare
    WRONG_TYPE         argument value has the wrong JSON type
    BAD_ENUM           value not in the declared enum
    PATH_ESCAPE        a path argument contains '..' (the harness blocks it)
    REPEATED_CALL      identical call repeated (set by loops.py, not here)
"""

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Any

from .schemas import ToolRegistry


class CallError(str, Enum):
    NO_CALL = "NO_CALL"
    TRUNCATED_CALL = "TRUNCATED_CALL"
    MALFORMED_SYNTAX = "MALFORMED_SYNTAX"
    UNKNOWN_TOOL = "UNKNOWN_TOOL"
    MISSING_ARG = "MISSING_ARG"
    UNEXPECTED_ARG = "UNEXPECTED_ARG"
    WRONG_TYPE = "WRONG_TYPE"
    BAD_ENUM = "BAD_ENUM"
    PATH_ESCAPE = "PATH_ESCAPE"
    REPEATED_CALL = "REPEATED_CALL"


@dataclass
class Issue:
    code: CallError
    detail: str = ""
    arg: str | None = None

    def to_dict(self) -> dict[str, Any]:
        return {"code": self.code.value, "detail": self.detail, "arg": self.arg}


PATH_ARG_NAMES = {"filepath", "path", "file_path", "filename", "file"}


def type_ok(value: Any, expected: str | list[str]) -> bool:
    kinds = expected if isinstance(expected, list) else [expected]
    for t in kinds:
        if t == "string" and isinstance(value, str):
            return True
        if t == "integer" and isinstance(value, int) and not isinstance(value, bool):
            return True
        if t == "number" and isinstance(value, (int, float)) and not isinstance(value, bool):
            return True
        if t == "boolean" and isinstance(value, bool):
            return True
        if t == "array" and isinstance(value, list):
            return True
        if t == "object" and isinstance(value, dict):
            return True
        if t == "null" and value is None:
            return True
    return False


def validate_call(name: str, args: dict[str, Any], registry: ToolRegistry) -> list[Issue]:
    spec = registry.get(name)
    if spec is None:
        return [Issue(CallError.UNKNOWN_TOOL, f"'{name}' not in {registry.names}")]
    issues: list[Issue] = []
    for req in spec.required:
        if req not in args:
            issues.append(Issue(CallError.MISSING_ARG, f"'{req}' is required", req))
    for k, v in args.items():
        p = spec.params.get(k)
        if p is None:
            issues.append(Issue(CallError.UNEXPECTED_ARG, f"'{k}' not declared by {name}", k))
            continue
        if not type_ok(v, p.type):
            issues.append(Issue(CallError.WRONG_TYPE, f"'{k}' should be {p.type}, got {type(v).__name__}", k))
            continue
        if p.enum is not None and v not in p.enum:
            issues.append(Issue(CallError.BAD_ENUM, f"'{k}'={v!r} not in {p.enum}", k))
        if k in PATH_ARG_NAMES and isinstance(v, str) and ".." in v.replace("\\", "/").split("/"):
            issues.append(Issue(CallError.PATH_ESCAPE, f"'{k}' contains '..'", k))
    return issues


## 1. Settings

In [ ]:
N_TASKS = 5              # how many dev tasks to run
TASK_IDS = None          # or a list, e.g. ["fastapi_11194", "rich_3454"]
MODEL_HINT = "e4b"       # substring that picks the model folder under /kaggle/input
MODE = "strict"          # strict = like the official harness; guard = with toolguard repairs
TIME_MINUTES = 30        # per-task agent time budget (official default is 60)
MAX_TOOL_CALLS = 50
MAX_MODEL_LEN = 32768    # lower to 16384 if vLLM runs out of memory
THINKING = "default"     # "on" | "off" | "default"
RUN_GOLD_CHECK = True
OUT = "/kaggle/working/fl_runs"

## 2. Find inputs and GPUs

In [ ]:
import glob, json, os, subprocess, sys, time, urllib.request
sys.path.insert(0, os.path.abspath("faultlines/src"))
from faultlines.harness import DataLayout

layout = DataLayout.find("/kaggle/input")
print("dataset:", layout.root, "| tasks:", sum(1 for _ in open(layout.tasks)))
cfgs = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
        if MODEL_HINT in p.lower() and "snapshots" not in p]
assert cfgs, f"No model folder matching '{MODEL_HINT}' — add Gemma 4 as an input (see top cell)."
MODEL_DIR = os.path.dirname(sorted(cfgs, key=len)[0])
print("model:", MODEL_DIR)

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
print("GPUs:", gpus)
N_GPU = max(1, len(gpus))
DTYPE = "half" if any(x in " ".join(gpus) for x in ("T4", "P100", "V100")) else "bfloat16"
print("dtype:", DTYPE, "| tensor parallel:", N_GPU)

## 3. Install vLLM and Python 3.12

The task wheels target Python 3.12/3.13, so the per-task venvs use a Python 3.12 managed by `uv`.

In [ ]:
%%bash
pip install -q -U vllm uv 2>&1 | tail -3
uv python install 3.12 2>&1 | tail -1

In [ ]:
PY312 = subprocess.run(["uv", "python", "find", "3.12"], capture_output=True, text=True).stdout.strip()
print("task python:", PY312)
FL = [sys.executable, "-m", "faultlines.cli"]
ENV = {**os.environ, "PYTHONPATH": os.path.abspath("faultlines/src")}

def fl(*args):
    """Run a faultlines command, streaming its output."""
    p = subprocess.Popen(FL + list(args), env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

task_args = (["--task-ids", *TASK_IDS] if TASK_IDS else ["--limit", str(N_TASKS)])

## 4. Gold replay: environment check (no GPU needed)

Every task should resolve. Any failure here is an environment problem, not a model problem. Fix those before trusting model results.

In [ ]:
if RUN_GOLD_CHECK:
    t0 = time.time()
    fl("harness", "--data", str(layout.root), "--gold", *task_args, "--out", f"{OUT}/gold",
       "--python", PY312, "--online-fallback", "--time-minutes", "10")
    print(f"gold replay wall time: {time.time() - t0:.0f}s")

## 5. Start the vLLM server

In [ ]:
log = open(f"/kaggle/working/vllm.log", "w")
srv = subprocess.Popen([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_DIR,
                        "--served-model-name", "gemma", "--dtype", DTYPE, "--tensor-parallel-size", str(N_GPU),
                        "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.90", "--port", "8000"],
                       stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5); break
    except Exception:
        if srv.poll() is not None:
            print(open("/kaggle/working/vllm.log").read()[-4000:]); raise RuntimeError("vLLM exited")
        if time.time() - t0 > 1800:
            raise TimeoutError("vLLM did not start in 30 min")
        time.sleep(10)
print(f"vLLM ready in {time.time() - t0:.0f}s")

## 6. Chat template and tool-call check

**Look at this output once.** Check that tools are declared in the prompt, that the model's reply contains `<|tool_call>call:get_status{}<tool_call|>` (or similar), and that the tool result shows up in the second render.

In [ ]:
from faultlines.harness.model import CompletionEndpointModel
from faultlines.toolguard import ToolGuard, ToolRegistry
kw = {} if THINKING == "default" else {"enable_thinking": THINKING == "on"}
m = CompletionEndpointModel("http://127.0.0.1:8000", "gemma", MODEL_DIR, max_model_len=MAX_MODEL_LEN,
                            chat_template_kwargs=kw)
tools = [s.to_declaration() for s in ToolRegistry.default_harness().specs.values()]
msgs = [{"role": "system", "content": "You fix bugs using tools."},
        {"role": "user", "content": "Call the get_status tool now."}]
print("---- prompt tail ----\n", m.render(msgs, tools)[-1200:])
t = m.generate(msgs, tools)
print("---- raw reply ----\n", repr(t.text[:1500]), "| finish:", t.finish_reason)
print("---- guard ----\n", ToolGuard(ToolRegistry.default_harness()).check(t.text).to_dict())
msgs2 = msgs + [{"role": "assistant", "content": "", "tool_calls": [{"type": "function",
                 "function": {"name": "get_status", "arguments": {}}}]},
                {"role": "tool", "name": "get_status", "content": json.dumps({"tool_calls_used": 0})}]
print("---- render with tool result ----\n", m.render(msgs2, tools)[-900:])

## 7. Run the model on N tasks

In [ ]:
t0 = time.time()
fl("harness", "--data", str(layout.root), *task_args, "--out", f"{OUT}/model",
   "--endpoint", "http://127.0.0.1:8000", "--model", "gemma", "--tokenizer", MODEL_DIR,
   "--mode", MODE, "--time-minutes", str(TIME_MINUTES), "--max-tool-calls", str(MAX_TOOL_CALLS),
   "--max-model-len", str(MAX_MODEL_LEN), "--thinking", THINKING, "--python", PY312, "--online-fallback",
   "--label", f"{os.path.basename(MODEL_DIR)}/{MODE}")
MODEL_WALL = time.time() - t0
print(f"model run wall time: {MODEL_WALL:.0f}s")

## 8. Timing and how many runs fit in your GPU quota

In [ ]:
runs = [json.loads(l) for l in open(f"{OUT}/model/runs.jsonl")]
print(f"{'task':24s} {'resolved':>8s} {'exit':>12s} {'turns':>5s} {'setup_s':>8s} {'agent_s':>8s} {'verify_s':>8s}")
for r in runs:
    tm = r["timing"]
    print(f"{r['task_id']:24s} {str(r['resolved']):>8s} {r['exit_reason']:>12s} {len(r['steps']):5d} "
          f"{tm.get('setup_s', 0):8.0f} {tm.get('agent_s', 0):8.0f} {tm.get('verify_s', 0):8.0f}")
per_run = MODEL_WALL / max(1, len(runs))
print(f"\nmean wall time per run: {per_run / 60:.1f} min")
for hours in (30, 60):
    print(f"runs that fit in {hours} GPU-hours: ~{int(hours * 3600 / per_run)}")

## 9. Label and summarize

In [ ]:
fl("label", f"{OUT}/model/runs.jsonl", "-o", f"{OUT}/model/labeled.jsonl")
fl("summarize", f"{OUT}/model/labeled.jsonl")
fl("replay", f"{OUT}/model/labeled.jsonl")

In [ ]:
# Peek at the raw model text of the first run (what the paper's step labels are built from)
r = [json.loads(l) for l in open(f"{OUT}/model/labeled.jsonl")][0]
print(r["task_id"], r["primary_failure"], r["secondary"], r["evidence"])
for s in r["steps"][:6]:
    print(f"\n--- turn {s['index']} | {s['call_status']} {s['call_issues']} | tool={s['tool_name']} | events={s['events']}")
    print(s["raw_output"][-800:])

In [ ]:
srv.terminate()
print("done — outputs in", OUT)